# 알약 검출 데이터 EDA (Kaggle 대회 원본 + AI Hub 조합 데이터)

## EDA의 목적
- 알약 사진에서 알약의 위치와 종류를 찾는 object detection 모델을 학습하기에 앞서, 데이터의 형태와 분포를 확인하고, 최적의 모델 학습을 위한 인사이트를 획득한다.
- 본 EDA를 통해 라벨 정제, 입력 크기, 증강, 분할 방식을 확정하고, AI Hub 데이터로 무엇을 얼마나 보충할지 정한다.
- 각 섹션의 분석 결과를 라벨 정제, 입력 크기, 데이터 증강, 후처리, 보충 데이터, 분할, 평가 방법에 대한 결정으로 연결하고, 9번 EDA 결론에서 결정 사항 표로 정리한다.
- 기준 데이터는 대회 원본 `data/raw/`이다. 보충 데이터는 `data/external/aihub/`에 받는다.
- 노트북은 두 단계로 실행한다.
    - Phase A: 대회 원본과 AI Hub 조합 라벨 zip 8개(약 59MB)만으로 실행한다. (0~6-7번, 8번)
        - 라벨 zip은 `uv run python script/download_aihub.py 66065 66067 66068 66069 66070 66071 66072 66243`로 `data/external/aihub/train_annotations/`에 받는다. (README의 AI Hub 외부 데이터 받기)
    - Phase B: 6-6에서 고른 조합 이미지를 받은 뒤 실행한다. (6-8번 이후, 7번)

## EDA 목차
> 각 분석 결과 아래에는 **인사이트 / 결론** 마크다운 셀을 두어, 분석 결과와 그에 따른 결정을 기록한다.

0. 사전 설정
    - seed, 데이터 경로, 그림 저장 경로, 한글 폰트, 도우미 함수를 정의하고 라벨을 이미지 단위로 읽는다.
1. 데이터셋 훑어보기
    - **파일과 라벨 구조**
        - 이미지와 annotation JSON 개수, 이미지당 JSON 수, 메타데이터 필드별 값 종류를 확인한다.
            - 목적: 라벨 로더를 어떻게 만들지와, 메타데이터 중 학습에 쓸 것이 있는지 정한다.
    - **train과 test의 입력 비교**
        - train과 test의 채널별 평균 밝기 분포를 비교한다.
            - 목적: test가 train과 같은 촬영 조건인지 확인하고, 색과 밝기 증강 범위를 정한다.
    - **이미지 육안 확인**
        - train 무작위 4장에 box를 겹쳐 그리고, test 무작위 4장과 나란히 본다. (seed 고정)
            - 목적: box가 알약을 제대로 감싸는지와 test에 train과 다른 장면이 있는지 확인한다.
2. 라벨 품질: 파일 명과 라벨이 잘 맞는 지.
    - **라벨 누락과 클래스 불일치**
        - 파일명에 적힌 K-코드와 bbox의 category_id를 대조해, 
            - 파일명에는 있는데 bbox에 없는 K-코드가 있는 지 점검한다.
            - bbox에는 있는데 파일 명에는 없는 category_id가 있는 지 점검한다.
    - **좌표 오류**
        - 이미지 밖으로 나간 bbox, area가 너비×높이와 다른 bbox, 다른 클래스와 좌표가 똑같은 bbox를 찾는다.
            - 찾은 box를 사진에 겹쳐 그려, 틀린 값과 고칠 값을 정한다.
    - **클래스별 crop 갤러리**
        - 모든 bbox 영역을 클래스별로 잘라 나란히 보고, 잘못된 클래스와 헐거운 box를 찾는다.
    - **train-test 중복**
        - 파일 내용이 완전히 같은 이미지를 찾는다.
    - **라벨 정제 목록**
        - 2-1~2-3에서 찾은 라벨 오류를 이미지별로 모으고 처리 방법을 정한다.
            - 목적: 지우거나 고칠 라벨과 이미지를 정한다.
            - 참고: 평가 지표 mAP@[0.75:0.95]는 IoU 0.75 미만인 box를 오답으로 처리하므로, box 오차에 매우 민감하다.
3. bbox 분포
    - **객체 수와 크기**
        - 이미지당 객체 수, box 크기와 종횡비, 입력 640으로 줄였을 때의 크기를 확인한다.
            - 목적: 학습 입력 크기를 정한다.
    - **위치와 겹침**
        - box 중심 위치와, 같은 이미지 안 box끼리의 IoU를 확인한다.
            - 목적: 후처리의 NMS IoU 임계값과 최대 검출 수를 정한다.
4. 클래스 분포와 각도 묶음
    - **클래스별 객체 수와 등장 조합 수**
        - 클래스마다 객체 수와, 그 클래스가 나오는 조합 수를 비교한다.
    - **조합당 이미지 수**
        - 조합 하나에 사진이 몇 장인지와 촬영 각도 분포를 확인한다.
    - **같은 조합 사진 비교**
        - 같은 조합의 각도별 사진을 나란히 본다.
            - 목적: 분할 단위(이미지 또는 조합), 클래스 불균형 대응, 회전과 반전 증강 사용 여부를 정한다.
5. test 알약 직접 확인: test에는 있고 train에는 없는 클래스가 있는 지 확인
    - **찾아보기용 train 갤러리**
        - 56개 클래스를 색과 모양으로 묶고, 클래스마다 crop과 K-코드, 제품명, 앞뒤 각인을 붙인다.
    - **test 확인 시트**
        - test 842장을 한 장에 12장씩 시트로 저장하고, 갤러리와 대조해 처음 보는 알약을 기록표에 적는다.
    - **처음 보는 알약 목록**
        - 기록표를 읽어 처음 보는 알약을 각인별로 묶고, 나오는 test 사진 수를 센다.
            - 목적: test에만 있는 클래스를 확정해 보충 범위와 클래스 체계를 정한다.
            - 참고: test는 라벨이 없어서 클래스를 직접 셀 수 없으므로 사진을 직접 본다. 1차 확인은 Claude Code가 하고, 2차 확인은 사람과 Claude Code가 나눠서 한다.
6. AI Hub 조합 데이터로 보충
    - **파일 구성과 금지 범위**
        - AI Hub 576 데이터셋의 파일 목록에서 쓸 수 있는 파일과 금지된 파일을 정리한다.
    - **라벨 구조 비교**
        - AI Hub 조합 라벨과 Kaggle 라벨의 폴더 구조, JSON 키, id 값, 클래스 표기를 비교한다.
        - 같은 알약의 Kaggle JSON과 AI Hub JSON을 키마다 나란히 놓고 본다.
    - **출처 확인**
        - train 이미지와 라벨이 누락된 8장이 허용된 zip에 있는지 확인한다.
    - **클래스 커버리지**
        - 허용된 조합 라벨에서 train 클래스와 train에 없는 클래스의 객체 수를 센다.
    - **test에서 읽은 각인 대조**
        - 5-3 기록표의 각인을 AI Hub 라벨의 각인 필드에서 찾는다.
    - **보충 목록과 용량**
        - 클래스별 목표 객체 수에 따라 받을 zip, 꺼낼 이미지, 남길 용량을 계산하고 목록을 저장한다.
    - **보충 전후 분포**
        - 고른 보충 이미지의 라벨로, 클래스별 객체 수와 등장 조합 수를 보충 전후로 비교한다. (여기까지 다운로드 전)
    - **중복 검사, test 전용 클래스 확정, Kaggle 형식 변환, 조합에 없는 클래스** (Phase B)
            - 목적: 보충할 데이터와 변환 방법을 정한다.
7. 분할 만들기 (Phase B)
    - 원본과 보충 데이터를 합쳐 조합 단위로 train과 val을 나눈다.
        - 목적: 팀 전체가 같은 val로 모델을 비교하게 한다.
8. 평가 기준선
    - 사전 준비: `uv add pycocotools`로 COCO 평가 코드를 추가했다. (`pyproject.toml`과 `uv.lock`은 함께 커밋)
    - **mAP 계산 검증**
        - 정답 box를 예측으로 넣었을 때 mAP@[0.75:0.95]가 1.0인지 확인한다.
    - **box 오차에 대한 민감도**
        - 정답 box를 n픽셀 옮겼을 때 mAP가 얼마나 떨어지는지 본다.
    - **제출 category_id 체계**
        - 제출 파일에 넣을 category_id가 어떤 값인지 확인한다.
            - 목적: 로컬 평가 코드를 검증하고, 제출 형식과 라벨 정밀도의 기준을 정한다.
9. EDA 결론
    - 1~8번의 결과를 라벨 정제, 입력 크기, 데이터 증강, 후처리, 보충 데이터, 분할, 평가 방법 항목별 결정 사항 표로 정리한다.

## 0. 사전 설정

In [ ]:
import collections
import contextlib
import csv
import hashlib
import io
import json
import os
import sys
import unicodedata
import zipfile
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display
from matplotlib import font_manager
from PIL import Image, ImageDraw, ImageFont
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

from pill_detection.data import combo_name, load_annotations, pill_codes, read_aihub_label_zip
from pill_detection.utils import draw_boxes

# 실행 환경의 버전 확인 (팀원 간 환경 차이로 오류가 생겼을 때 원인 파악용)
print(f"Python       : {sys.version.split()[0]}")
print(f"numpy        : {np.__version__}")
print(f"matplotlib   : {matplotlib.__version__}")

In [ ]:
# 랜덤 샘플 시각화 등의 재현성을 위한 seed 고정
SEED = 42
np.random.seed(SEED)

# 노트북을 어디서 실행해도 레포 최상위(pyproject.toml이 있는 폴더)를 기준으로 경로를 잡는다
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
TRAIN_IMG_DIR = RAW_DIR / 'train_images'
TRAIN_ANN_DIR = RAW_DIR / 'train_annotations'
TEST_IMG_DIR = RAW_DIR / 'test_images'
AIHUB_DIR = PROJECT_ROOT / 'data' / 'external' / 'aihub'
# 그림 저장 경로 (runs/는 git에서 제외됨)
FIG_DIR = PROJECT_ROOT / 'runs' / 'eda_raewoo' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# 대회 이미지 크기 (너비, 높이)
IMG_W, IMG_H = 976, 1280

# 그래프의 한글이 깨지지 않도록 설치된 한글 폰트 사용 (macOS, Windows, Linux 순)
installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
for font_name in ['AppleGothic', 'Malgun Gothic', 'NanumGothic']:
    if font_name in installed_fonts:
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False  # 한글 폰트에서 마이너스 기호가 깨지는 문제 방지


def save_fig(fig, name):
    """그림을 FIG_DIR에 png 파일로 저장한다."""
    fig.savefig(FIG_DIR / f'{name}.png', dpi=120, bbox_inches='tight')


def show_table(header, rows):
    """열 이름 리스트와 행 리스트를 마크다운 표로 출력한다."""
    lines = ['| ' + ' | '.join(header) + ' |', '|' + ' --- |' * len(header)]
    # 값 속 |(예: 분할선 각인 K|50)가 칸 구분자로 읽히지 않게 \|로 바꾼다
    lines += ['| ' + ' | '.join(str(value).replace('|', r'\|') for value in row) + ' |' for row in rows]
    display(Markdown('\n'.join(lines)))


print(f"레포 최상위  : {PROJECT_ROOT}")
print(f"그래프 폰트  : {plt.rcParams['font.family'][0]}")
print(f"그림 저장    : {FIG_DIR}")

In [ ]:
def box_iou(box_a, box_b):
    """두 xywh box의 IoU(겹친 넓이 ÷ 합친 넓이, 1이면 완전히 같음)를 계산한다."""
    inter_w = max(0, min(box_a[0] + box_a[2], box_b[0] + box_b[2]) - max(box_a[0], box_b[0]))
    inter_h = max(0, min(box_a[1] + box_a[3], box_b[1] + box_b[3]) - max(box_a[1], box_b[1]))
    inter = inter_w * inter_h
    return inter / (box_a[2] * box_a[3] + box_b[2] * box_b[3] - inter)


def crop_square(image, box, pad=0.1, size=224):
    """box 중심에서 긴 변보다 pad만큼 큰 정사각형을 잘라 size×size로 줄인다. 알약 모양이 늘어나지 않는다."""
    x, y, w, h = box
    side = max(w, h) * (1 + pad)
    cx, cy = x + w / 2, y + h / 2
    return image.crop((int(cx - side / 2), int(cy - side / 2), int(cx + side / 2), int(cy + side / 2))).resize((size, size))

In [ ]:
train_records = load_annotations(TRAIN_ANN_DIR)
train_files = sorted(os.listdir(TRAIN_IMG_DIR))
test_files = sorted(os.listdir(TEST_IMG_DIR), key=lambda name: int(name.split('.')[0]))

# 객체(알약 하나) 단위 목록: (이미지 파일명, K-코드, xywh bbox)
train_objects = [(fn, ann['category_id'], ann['bbox'])
                 for fn, record in sorted(train_records.items()) for ann in record['annotations']]
train_classes = sorted({code for _, code, _ in train_objects})
train_combos = sorted({combo_name(fn) for fn in train_files})

assert set(train_records) == set(train_files), '라벨이 있는 이미지와 이미지 파일 목록이 다르다'
print(f'train 이미지 {len(train_files)}장, 객체(bbox) {len(train_objects)}개, 클래스 {len(train_classes)}개, 조합 {len(train_combos)}개')
print(f'test 이미지 {len(test_files)}장')

In [ ]:
# train_records 예시
image_file_name = "K-001900-016548-019607-029451_0_2_0_2_70_000_200.png"
record = train_records[image_file_name]
print(f"이미지파일 {image_file_name}의 이름에서 알 수 있는 알약 개수: {len(pill_codes(image_file_name))}개")
print(f"이미지 파일 {image_file_name}의 등록된 정답 알약 개수: {len(record['annotations'])}개")
print(f"이미지파일 {image_file_name}의 정답 bbox 좌표: ")
for idx, annotation in enumerate(record['annotations']):
    print(f"{idx}번째 알약 \ncategory_id: {annotation['category_id']}\n이름: {annotation['image']['dl_name']}\n위치: {annotation['bbox']}")
      

## 1. 데이터셋 훑어보기

### 1-1. 파일과 라벨 구조
- 이미지 수, annotation JSON 수, 이미지당 JSON 수를 센다.
- 이미지 메타데이터 필드별로 어떤 값이 있는지 센다.
    - 조합(combo)은 함께 찍은 알약 3~4개의 묶음이다. 이미지 파일명의 첫 `_` 앞부분이 조합 이름이다. (`K-001900-016548-019607-029451`)
- 목적: 라벨 로더를 어떻게 만들지와, 메타데이터 중 학습에 쓸 것이 있는지 정한다.

In [ ]:
json_count = sum(1 for _ in TRAIN_ANN_DIR.rglob('*.json'))
anns_per_image = collections.Counter(len(r['annotations']) for r in train_records.values())
test_sizes = collections.Counter(Image.open(TEST_IMG_DIR / fn).size for fn in test_files)  # 파일 머리말만 읽어 빠르다
train_sizes = collections.Counter((r['image']['width'], r['image']['height']) for r in train_records.values())
same_code = sum(ann['category_id'] == int(ann['image']['drug_N'][2:]) for r in train_records.values() for ann in r['annotations'])


def value_counts(field):
    """annotation JSON마다 붙은 이미지 메타데이터 필드의 값 분포를 '값 개수' 문자열로 만든다."""
    counts = collections.Counter(ann['image'][field] for r in train_records.values() for ann in r['annotations'])
    return ', '.join(f'{value} {count}' for value, count in counts.most_common())


all_fields = list(next(iter(train_records.values()))['image'].keys())
# 0은 빈 값으로 보지 않는다. camera_lo처럼 0이 실제 값인 필드가 있다.
empty_fields = [f for f in all_fields
                if all(ann['image'][f] in (None, '') for r in train_records.values() for ann in r['annotations'])]

rows = [
    ['train 이미지 / annotation JSON', f'{len(train_files)}장 / {json_count}개'],
    ['이미지당 JSON(=알약) 수', ', '.join(f'{k}개 {v}장' for k, v in sorted(anns_per_image.items()))],
    ['클래스(category_id) 수', f'{len(train_classes)}개'],
    ['category_id = drug_N 숫자인 annotation', f'{same_code}/{json_count}'],
    ['조합 수', f'{len(train_combos)}개'],
    ['test 이미지', f'{len(test_files)}장'],
    ['해상도 (너비×높이)', f'train {dict(train_sizes)}, test {dict(test_sizes)}'],
    ['drug_S (알약 상태)', value_counts('drug_S')],
    ['back_color (배경)', value_counts('back_color')],
    ['light_color (조명)', value_counts('light_color')],
    ['drug_dir (알약 면)', value_counts('drug_dir')],
    ['camera_la (카메라 위도, 도)', value_counts('camera_la')],
    ['camera_lo (카메라 경도, 도)', value_counts('camera_lo')],
    ['\'image\' 안에서 모든 값이 비어 있는 필드', f'{len(empty_fields)}개 / 전체 {len(all_fields)}개: {", ".join(empty_fields)}'],
]
show_table(['항목', '값'], rows)

**인사이트**
- train은 232장, annotation JSON은 763개이다. JSON 하나가 알약 하나의 라벨이다.
    - 이미지 하나의 라벨이 알약 수만큼 2~4개 JSON으로 흩어져 있다. (2개 7장, 3개 151장, 4개 74장)
- 클래스는 56개이고, category_id는 763개 모두 K-코드 숫자(drug_N)와 같다.
- train 조합은 114개이다. test는 842장으로 train의 3.6배이다.
- 해상도는 train과 test 모두 976×1280 한 가지이다.
- 촬영 조건 필드는 거의 값이 하나뿐이다.
    - drug_S는 모두 정상알약, back_color는 모두 연회색 배경, light_color는 모두 주백색, camera_lo는 모두 0이다.
    - drug_dir는 앞면 751, 뒷면 12이다.
- camera_la는 70°, 75°, 90° 세 가지로 고르게 나뉜다. (237, 249, 277)
- 'image' 필드 50개 중 4개는 모든 값이 비어 있다.
    - `mark_code_back_anal`, `mark_code_back_img`, `mark_code_front`, `mark_code_back`(알약 마크의 형태, 이미지, 코드)이다.
    - 값은 763개 모두 빈 문자열(745개)이나 `None`(18개)이다.

**결론**
- 라벨은 `load_annotations`(공용 모듈)로 이미지 단위로 합쳐 읽는다.
    - YOLO 라벨 파일은 이미지 하나당 파일 하나라서, 변환할 때도 이 함수의 결과를 쓴다.
- 클래스 번호는 K-코드 숫자를 기준으로 하고, YOLO 학습용 0부터 시작하는 인덱스와의 매핑 표를 따로 만든다.
- 메타데이터는 학습 입력으로 쓰지 않는다.
    - 촬영 조건 필드는 값이 하나뿐이라 구분에 도움이 되지 않고, test 이미지에는 메타데이터가 없다.

### 1-2. train과 test의 입력 비교
- 이미지마다 R, G, B 채널의 평균 밝기를 구해 train과 test의 분포를 겹쳐 그린다.
    - 배경이 이미지 대부분을 차지하므로, 채널 평균은 배경색과 조명을 나타낸다.
    - 계산을 빠르게 하려고 이미지를 1/8 크기로 줄여서 구한다.
- 목적: test가 train과 같은 촬영 조건인지 확인하고, 색과 밝기 증강 범위를 정한다.

In [ ]:
def channel_means(image_dir, files):
    """이미지마다 R, G, B 채널 평균 밝기(0~255)를 구한다."""
    return np.array([np.asarray(Image.open(image_dir / fn).convert('RGB').reduce(8)).reshape(-1, 3).mean(axis=0)
                     for fn in files])


train_rgb = channel_means(TRAIN_IMG_DIR, train_files)
test_rgb = channel_means(TEST_IMG_DIR, test_files)

bins = np.linspace(min(train_rgb.min(), test_rgb.min()), max(train_rgb.max(), test_rgb.max()), 41)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True, layout='constrained')
for ch, (ax, name, color) in enumerate(zip(axes, ['R', 'G', 'B'], ['tab:red', 'tab:green', 'tab:blue'])):
    # density=True: 막대 면적의 합을 1로 맞춰, 개수가 다른 train(232장)과 test(842장)를 같은 눈금으로 비교
    ax.hist(train_rgb[:, ch], bins=bins, density=True, alpha=0.5, color=color, label='train')
    ax.hist(test_rgb[:, ch], bins=bins, density=True, histtype='step', linewidth=2, color='black', label='test')
    ax.set_title(f'{name} 채널\n평균 train {train_rgb[:, ch].mean():.1f} / test {test_rgb[:, ch].mean():.1f}')
    ax.set_xlabel(f'이미지별 {name} 평균 (0~255)')
axes[0].set_ylabel('밀도')
axes[0].legend()
fig.suptitle('이미지별 채널 평균 밝기 분포: train(막대)과 test(선) 비교')
save_fig(fig, '1-2_channel_means')
plt.show()

**인사이트**
- 세 채널 모두 test 평균이 train보다 낮다: G 121.1(train 129.7), R 111.5(train 115.6), B 144.7(train 148.7)
- 차이가 가장 큰 G 채널에서 train은 100~165에 넓게 퍼져 있지만, test는 100~120에 몰려 있다.
- test 분포의 대부분은 train 범위(약 90~175) 안에 들어온다.

**결론**
- test는 train과 같은 촬영 장비와 배경으로 찍은 사진으로 보고, 별도 색 보정은 하지 않는다.
- 색과 밝기 증강은 ultralytics 기본값(`hsv_h=0.015`, `hsv_s=0.7`, `hsv_v=0.4`)을 그대로 쓴다.
    - 기본 밝기 증강 폭(±40%)이 train과 test의 채널 평균 차이(3~7%)보다 충분히 넓다.

### 1-3. 이미지 육안 확인
- train 무작위 4장에 bbox와 K-코드를 겹쳐 그리고, test 무작위 4장과 나란히 본다. (seed 고정)
- 목적: box가 알약을 제대로 감싸는지와 test에 train과 다른 장면이 있는지 확인한다.

In [ ]:
rng = np.random.default_rng(SEED)  # 셀을 다시 실행해도 같은 이미지가 뽑히도록 고정
fig, axes = plt.subplots(2, 4, figsize=(16, 10.5), layout='constrained')
for ax, fn in zip(axes[0], rng.choice(train_files, size=4, replace=False)):
    anns = train_records[fn]['annotations']
    ax.imshow(draw_boxes(TRAIN_IMG_DIR / fn, [a['bbox'] for a in anns], [a['category_id'] for a in anns], width=6))
    ax.set_title(f'train: {combo_name(fn)}\n각도 {fn.split("_")[5]}°', fontsize=10)
for ax, fn in zip(axes[1], rng.choice(test_files, size=4, replace=False)):
    ax.imshow(Image.open(TEST_IMG_DIR / fn))
    ax.set_title(f'test: {fn}', fontsize=10)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle('위: train 무작위 4장 (빨간 box = 라벨, 숫자 = K-코드) / 아래: test 무작위 4장')
save_fig(fig, '1-3_samples')
plt.show()

**인사이트**
- train 4장 모두 box가 알약을 빈틈없이 감싼다.
- test 4장도 train처럼 알약 3~4개를 띄워 놓고 위에서 찍은 사진이다.
- 배경은 두 가지로 보인다.
    - 위쪽 train 4장처럼 모서리가 하얗게 뜬 밝은 회청색 배경
    - 아래쪽 test 4장처럼 어두운 보라색 배경
- 1-2에서 test의 G 채널 평균이 100~120에 몰린 것은, test에 어두운 보라색 배경 사진이 많기 때문으로 보인다.
    - 무작위로 뽑은 test 4장은 모두 어두운 보라색 배경이고, train 4장은 모두 밝은 회청색 배경이다.

**결론**
- test는 train과 같은 촬영 방식이므로, 입력 전처리를 train과 test에 똑같이 쓴다.

## 2. 라벨 품질
- 대회 설명(DATA.md)은 제공 데이터에 크고 작은 오류가 있다고 예고했다.
- 목적: 지우거나 고칠 라벨과 이미지를 정한다.
    - 평가 지표 mAP@[0.75:0.95]는 IoU 0.75 미만인 box를 오답으로 처리한다. 라벨 box가 조금만 틀려도 학습과 평가가 모두 흔들린다.

### 2-1. 라벨 누락과 클래스 불일치
- 이미지마다 파일명에 적힌 K-코드와 annotations의 category_id를 대조한다.
    - 파일명 `K-003351-035206-041768_...`는 이 사진에 알약 3개(K-003351, K-035206, K-041768)가 있다는 뜻이다.
- 라벨 누락: 파일명에는 있는데 bbox가 없는 K-코드를 찾는다.
- 클래스 불일치: bbox에는 붙어 있는데 파일명에 없는 K-코드를 찾는다.
    - 같은 K-코드가 파일명보다 많이 붙은 경우도 여기에 센다.
- 이 대조는 K-코드 목록만 비교한다. box 안의 알약이 실제로 그 클래스인지는 이 대조로 알 수 없다.

In [ ]:
train_object_count = collections.Counter(code for _, code, _ in train_objects)
missing_files = []  # 파일명의 알약 중 bbox가 없는 이미지 (라벨 누락)
extra_files = []  # 파일명에 없는 K-코드가 붙은 이미지 (클래스 불일치)
rows = []
for fn in train_files:
    named = collections.Counter(pill_codes(fn))
    labeled = collections.Counter(a['category_id'] for a in train_records[fn]['annotations'])
    # Counter 뺄셈은 개수까지 비교한다. 같은 K-코드가 파일명보다 많이 붙으면 extra에 들어간다.
    missing = sorted((named - labeled).elements())
    extra = sorted((labeled - named).elements())
    if missing:
        missing_files.append(fn)
    if extra:
        extra_files.append(fn)
    if missing or extra:
        rows.append([f'`{fn}`', len(pill_codes(fn)), len(train_records[fn]['annotations']),
                     ', '.join(f'K-{c:06d}' for c in missing) or '-',
                     ', '.join(f'K-{c:06d}' for c in extra) or '-',
                     ', '.join(str(train_object_count[c]) for c in missing) or '-'])
rows.append(['**합계**', '', '', f'**{len(missing_files)}장**', f'**{len(extra_files)}장**', ''])
show_table(['이미지', '파일명 알약 수', 'bbox 수', 'bbox가 없는 K-코드 (누락)', '파일명에 없는 K-코드 (불일치)',
            '누락 클래스의 train 객체 수'], rows)

**인사이트**
- 8장(train의 3.4%)에서 파일명에 적힌 알약 하나씩의 bbox가 없다.
    - 8장은 아래 그림에서 사진으로 확인한다.
- 빠진 알약은 K-003351이 4장으로 가장 많다.
- K-004543은 train 객체가 5개뿐인데, 그중 한 장에서 라벨이 빠졌다.
- 라벨이 빠진 알약을 그대로 학습하면, 모델은 그 알약을 배경으로 배운다.
- 파일명에 없는 K-코드가 붙은 이미지는 0장이다.
    - 763개 annotation의 category_id가 모두 그 이미지 파일명에 있는 K-코드이다.
    - 한 이미지 안에 같은 K-코드가 두 번 붙은 경우도 없다.

**결론**
- 8장은 빠진 알약의 box를 사진에서 그려 넣어 쓴다.
    - 라벨에는 빠진 알약의 좌표가 없으므로, 사진에서 알약 범위를 재서 box를 만든다.
    - 평가 지표가 IoU 0.75~0.95를 요구하므로(2절 소개), 그리는 방법이 라벨 box만큼 정확한지 아래에서 먼저 확인한다.
- 파일명을 기준으로 category_id를 고칠 라벨은 없다.
    - 클래스가 잘못 붙은 box는 box 안의 알약을 직접 봐야 찾을 수 있다.

- 라벨이 누락된 8장에 라벨 box를 모두 겹쳐 그린다.
    - box 위 글자는 K-코드와 메타데이터의 각인(print_front/print_back)이다. 이 글자를 box 안 알약에 찍힌 글자와 대조하면, box가 어느 알약을 감싸는지 알 수 있다.
    - 그림 제목 둘째 줄은 파일명에는 있지만 box가 없는 알약의 K-코드와 각인이다.
- 빠진 알약이 사진에 실제로 있는지, box가 없는 알약이 어느 것인지 본다.
- 클래스 불일치는 0장(2-1 표)이라 따로 그릴 이미지가 없다.
    - 대신 box가 있는 알약의 각인을 box의 K-코드와 대조해, 클래스가 맞게 붙었는지 사진으로 확인한다.

In [ ]:
def pill_label(ann):
    """box 위에 적을 K-코드와 각인(메타데이터 print_front/print_back)을 만든다."""
    marks = [ann['image'][key] for key in ('print_front', 'print_back') if ann['image'].get(key)]
    return f"K-{ann['category_id']:06d} ({'/'.join(marks) or '각인 없음'})"


class_example = {}  # K-코드 → 그 클래스의 annotation 하나 (bbox가 없는 알약의 각인을 적는 데 쓴다)
for fn in train_files:
    for ann in train_records[fn]['annotations']:
        class_example.setdefault(ann['category_id'], ann)

fig, axes = plt.subplots(2, 4, figsize=(24, 18), layout='constrained')
for ax, fn in zip(axes.flat, missing_files):
    ax.imshow(Image.open(TRAIN_IMG_DIR / fn))
    for ann in train_records[fn]['annotations']:
        x, y, w, h = ann['bbox']
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=2.5))
        ax.text(x, y - 8, pill_label(ann), fontsize=9, va='bottom', bbox=dict(facecolor='lime', alpha=0.85, pad=1.5, linewidth=0))
    labeled = collections.Counter(a['category_id'] for a in train_records[fn]['annotations'])
    missing = (collections.Counter(pill_codes(fn)) - labeled).elements()
    ax.set_title(f"{fn}\nbbox 없음: {', '.join(pill_label(class_example[c]) for c in missing)}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(f'라벨이 누락된 이미지 {len(missing_files)}장 (연두 box = 라벨에 있는 box, 글자 = K-코드(각인), 제목 둘째 줄 = 파일명에는 있지만 box가 없는 알약)',
             fontsize=14)
save_fig(fig, '2-1b_missing_labels')
plt.show()

**인사이트**
- 8장 모두 파일명에 적힌 알약이 사진에 다 있지만, 그중 하나에 box가 없다.
    - K-003351(HT 정제) 4장: 모두 70° 사진이고, 아래쪽 가운데의 주황 원형 HT 정제에 box가 없다.
    - K-020014: 왼쪽 아래의 큰 갈색 원형 정제(십자 분할선)에 box가 없다.
    - K-032310(글리아타민): 오른쪽 아래의 노란 타원형 연질캡슐에 box가 없다.
    - K-033880(JIC): 왼쪽 아래의 노란 장방형 정제에 box가 없다.
    - K-004543(Airtal): 오른쪽 위의 흰 원형 정제에 box가 없다.
- box가 있는 알약 17개는 모두 box의 K-코드와 알약에 찍힌 각인이 맞는다.
    - K-016548은 메타데이터에 각인이 비어 있다. 같은 사진의 나머지 세 box가 각인과 맞으므로, 남은 GB 정제가 K-016548이다.
    - 이 8장에는 클래스가 잘못 붙은 box가 없다. 2-1 표의 클래스 불일치 0장과 같다.

**결론**
- 빠진 알약이 사진에 그대로 있으므로, box 없이 학습에 넣으면 모델이 그 알약을 배경으로 배운다. 빠진 box는 아래에서 그려 넣는다.
- box가 있는 알약은 클래스가 맞으므로, 8장에서 고칠 category_id는 없다.

- 빠진 box를 사진에서 그려 넣는 방법을 정하고, 라벨이 있는 box로 정확도를 먼저 확인한다.
    - 알약 범위: 영역 테두리 4픽셀의 중앙값을 바닥 색으로 보고, 아래 두 조건 중 하나를 만족하는 픽셀을 알약으로 센다.
        - 파랑 비율 B/(R+G+B)가 바닥보다 0.04 이상 낮다. 바닥은 푸르스름하고, 알약은 노랑, 주황, 갈색처럼 덜 푸르다.
        - 밝기 R+G+B가 바닥보다 20% 이상 높다. 흰 알약은 바닥보다 밝다.
        - 그림자는 바닥보다 어둡고 더 푸르러서 알약으로 세지 않는다.
    - 알약으로 센 픽셀이 가장 많은 행(열)의 10% 이상인 행과 열만 범위에 넣어, 바닥의 작은 잡음 점을 뺀다.
    - box: 알약 범위의 네 변에, 같은 클래스 라벨 box의 여백(box 변과 알약 범위 사이 거리) 중앙값을 더한다.
- 검증: 라벨 누락이 없는 224장의 box마다, box를 사방 20픽셀 넓힌 영역에서 알약 범위를 잰다.
    - 자기 box를 뺀 같은 클래스의 여백 중앙값으로 box를 다시 만들어, 라벨 box와의 IoU(겹친 넓이 ÷ 합친 넓이, 1이면 똑같음)를 본다.
    - 알약으로 센 픽셀이 영역의 5%보다 적으면 알약을 찾지 못한 것으로 보고 뺀다.
- 목적: 평가 지표가 IoU 0.75~0.95를 요구하므로(2절 소개), 그린 box가 라벨 box만큼 정확한지 확인하고 그려 넣을지 정한다.

In [ ]:
BLUE_DROP = 0.04  # 바닥보다 파랑 비율 B/(R+G+B)가 이만큼 낮으면 알약으로 센다 (그림자는 바닥보다 어둡고 더 푸르다)
BRIGHT_RISE = 0.2  # 바닥보다 이 비율 이상 밝으면 알약으로 센다 (흰 알약)
MIN_SHARE = 0.1  # 알약으로 센 픽셀 수가 가장 많은 행(열)의 이 비율 이상인 행(열)만 알약 범위에 넣는다 (바닥 잡음 제거)
MIN_AREA = 0.05  # 알약으로 센 픽셀이 영역의 이 비율보다 적으면 영역 안에 알약이 없다고 본다
REGION_PAD = 20  # 검증에서 라벨 box를 사방으로 이만큼 넓힌 영역에서 알약 범위를 잰다 (픽셀)


def pill_extent(image, region):
    """region [x0, y0, x1, y1] 안에서 알약이 차지하는 범위 [x0, y0, x1, y1]를 찾는다.

    region 테두리 4픽셀의 중앙값을 바닥 색으로 보고, 바닥보다 파랑 비율이 낮거나 밝은 픽셀을 알약으로 센다.
    region 안에는 알약이 하나만 있어야 하고, 알약으로 센 픽셀이 MIN_AREA보다 적으면 None을 돌려준다.
    """
    x0, y0, x1, y1 = region
    rgb = np.asarray(image.convert('RGB'), dtype=float)[y0:y1, x0:x1]
    total = rgb.sum(axis=2).clip(1)
    blue = rgb[..., 2] / total
    ring = np.zeros(total.shape, dtype=bool)
    ring[:4], ring[-4:], ring[:, :4], ring[:, -4:] = True, True, True, True
    mask = (blue < np.median(blue[ring]) - BLUE_DROP) | (total > np.median(total[ring]) * (1 + BRIGHT_RISE))
    if mask.mean() < MIN_AREA:
        return None
    rows = np.nonzero(mask.sum(axis=1) >= MIN_SHARE * mask.sum(axis=1).max())[0]
    cols = np.nonzero(mask.sum(axis=0) >= MIN_SHARE * mask.sum(axis=0).max())[0]
    return [x0 + int(cols[0]), y0 + int(rows[0]), x0 + int(cols[-1]) + 1, y0 + int(rows[-1]) + 1]


missing_codes = sorted({code for fn in missing_files  # 라벨이 누락된 알약의 클래스 (2-1 표)
                        for code in collections.Counter(pill_codes(fn)) - collections.Counter(a['category_id'] for a in train_records[fn]['annotations'])})

# 라벨 누락이 없는 이미지의 box마다 알약 범위를 재고, 라벨 box 변까지의 여백을 구한다
reference = []  # (클래스, 라벨 bbox, 알약 범위)
for fn in train_files:
    if fn in missing_files:
        continue
    image = Image.open(TRAIN_IMG_DIR / fn)
    for ann in train_records[fn]['annotations']:
        x, y, w, h = ann['bbox']
        if x >= IMG_W:  # 이미지 밖 bbox(2-2)는 뺀다
            continue
        region = [max(0, x - REGION_PAD), max(0, y - REGION_PAD), min(IMG_W, x + w + REGION_PAD), min(IMG_H, y + h + REGION_PAD)]
        reference.append((ann['category_id'], ann['bbox'], pill_extent(image, region)))
not_found = [bbox for _, bbox, ex in reference if ex is None]  # 영역 안에서 알약을 찾지 못한 box
reference = [item for item in reference if item[2] is not None]
margins = np.array([[ex[0] - x, ex[1] - y, x + w - ex[2], y + h - ex[3]] for _, (x, y, w, h), ex in reference])  # 왼, 위, 오른, 아래
reference_codes = np.array([code for code, _, _ in reference])
class_margins = {code: np.median(margins[reference_codes == code], axis=0) for code in set(reference_codes)}  # 클래스 → 여백 중앙값


def drawn_box(extent, margin):
    """알약 범위 [x0, y0, x1, y1]에 여백(왼, 위, 오른, 아래)을 더해 xywh box를 만든다."""
    left, top, right, bottom = margin
    return [int(np.round(v)) for v in (extent[0] - left, extent[1] - top, extent[2] - extent[0] + left + right, extent[3] - extent[1] + top + bottom)]


# 검증: 자기 box를 뺀 같은 클래스 box들의 여백 중앙값으로 box를 다시 만들어 라벨 box와 비교한다
# (같은 클래스에 다른 box가 없으면 전체 여백 중앙값을 쓴다)
ref_ious = []
for i, (code, bbox, extent) in enumerate(reference):
    others = (reference_codes == code) & (np.arange(len(reference)) != i)
    margin = np.median(margins[others] if others.any() else margins, axis=0)
    ref_ious.append(box_iou(drawn_box(extent, margin), bbox))
ref_ious = np.array(ref_ious)
is_missing_class = np.isin(reference_codes, missing_codes)

fig, axes = plt.subplots(1, 2, figsize=(17, 5.5), layout='constrained', gridspec_kw={'width_ratios': [1.2, 1]})
bins = np.linspace(0, 1, 51)
for mask, label, color in [(~is_missing_class, '나머지 클래스', 'tab:gray'), (is_missing_class, f'라벨이 누락된 알약의 {len(missing_codes)}개 클래스', 'tab:blue')]:
    axes[0].hist(ref_ious[mask], bins=bins, density=True, alpha=0.6, color=color,
                 label=f'{label} {mask.sum()}개 (중앙값 {np.median(ref_ious[mask]):.3f}, 최솟값 {ref_ious[mask].min():.3f})')
for t in (0.75, 0.95):
    axes[0].axvline(t, color='black', linestyle='--', linewidth=1)
axes[0].set_xlabel('다시 만든 box와 라벨 box의 IoU (1이면 똑같음)')
axes[0].set_ylabel('비율 (밀도)')
axes[0].set_title('라벨 box를 이 방법으로 다시 만들었을 때의 IoU (점선 = 평가 기준 0.75, 0.95)')
axes[0].legend(loc='upper left')
sides = ['왼쪽', '위', '오른쪽', '아래']
for k, side in enumerate(sides):
    axes[1].bar(np.arange(len(missing_codes)) + (k - 1.5) * 0.2, [class_margins[c][k] for c in missing_codes], width=0.2, label=side)
axes[1].set_xticks(range(len(missing_codes)), [f'K-{c:06d}\n({np.sum(reference_codes == c)}개)' for c in missing_codes])
axes[1].set_ylabel('라벨 box 변과 알약 범위 사이 여백 중앙값 (픽셀)')
axes[1].set_title('라벨이 누락된 알약 클래스의 라벨 여백 (괄호 = 여백을 잰 box 수)')
axes[1].legend(ncol=4, loc='upper left')
fig.suptitle(f'빠진 box를 그리는 방법의 검증 (라벨 누락이 없는 {len(train_files) - len(missing_files)}장의 box {len(reference)}개, '
             f'알약을 찾지 못한 box {len(not_found)}개 제외)', fontsize=14)
save_fig(fig, '2-1c_drawing_check')
plt.show()

**인사이트**
- 라벨이 누락된 알약의 5개 클래스는 다시 만든 box 199개의 IoU 중앙값이 0.974이고, 가장 낮은 것도 0.863이다.
    - 모두 평가 기준의 가장 낮은 값 0.75를 넘고, 94.5%는 0.95를 넘는다.
- 나머지 클래스의 box 534개도 IoU 중앙값이 0.963이다. 0.75보다 낮은 box는 1개(파랑 타원형 K-031885)뿐이다.
- 알약을 찾지 못한 box는 12개이다.
    - 초록 캡슐 K-002483 9개와 파랑 타원형 K-031885 2개는 알약도 푸르스름해서, 파랑 비율로 바닥과 구분되지 않는다.
    - 나머지 1개는 box를 넓힌 영역 안에 알약이 없다.
- 라벨 여백은 클래스마다 다르다.
    - K-003351은 8~10픽셀이고, K-020014와 K-033880은 12~25픽셀이다.
    - 5개 클래스 모두 오른쪽 여백이 가장 크거나 다른 변과 같다.

**결론**
- 빠진 box는 이 방법으로 그려 넣는다.
    - 빠진 알약은 주황, 갈색, 노랑, 흰색이라 바닥과 색이 확실히 다르다. (2-1의 사진 그림)
    - 여백은 클래스마다 다르므로, 같은 클래스 라벨 box의 여백 중앙값을 쓴다.
- 빠진 알약 하나만 들어가는 영역은 사진을 보고 정한다. 영역 안에 다른 알약이 들어가면 범위가 넓어진다.

- 라벨이 누락된 8장마다 빠진 알약 하나만 들어가는 영역(흰 점선)을 사진을 보고 정하고, 위의 방법으로 box(파란 점선)를 그린다.
    - 빠진 알약의 자리는 2-1의 사진 그림에서 확인한 것이다.
    - 칸 제목 셋째 줄은 같은 클래스 라벨 box의 긴 변×짧은 변 중앙값과, 위 검증에서 그 클래스의 IoU 중앙값이다.
        - 긴 알약은 놓인 방향에 따라 너비와 높이가 바뀌므로, 긴 변과 짧은 변으로 비교한다.
- 그린 box가 알약을 빈틈없이 감싸는지, 크기가 같은 클래스 라벨 box와 비슷한지 본다.

In [ ]:
MISSING_REGIONS = {  # (이미지, 빠진 K-코드) → 사진에서 눈으로 정한, 빠진 알약 하나만 들어가는 영역 [x0, y0, x1, y1]
    ('K-003351-013900-021325_0_2_0_2_70_000_200.png', 3351): [360, 810, 600, 1050],
    ('K-003351-013900-036637_0_2_0_2_70_000_200.png', 3351): [400, 830, 640, 1070],
    ('K-003351-020014-022074_0_2_0_2_90_000_200.png', 20014): [30, 700, 440, 1110],
    ('K-003351-021325-032310_0_2_0_2_90_000_200.png', 32310): [570, 800, 976, 1120],
    ('K-003351-032310-038162_0_2_0_2_70_000_200.png', 3351): [360, 840, 600, 1080],
    ('K-003351-033880-038162_0_2_0_2_75_000_200.png', 33880): [40, 570, 400, 1040],
    ('K-003351-035206-041768_0_2_0_2_70_000_200.png', 3351): [440, 850, 680, 1090],
    ('K-003544-004543-012247-016548_0_2_0_2_90_000_200.png', 4543): [620, 170, 860, 410],
}
assert [fn for fn, _ in MISSING_REGIONS] == missing_files
assert all((collections.Counter(pill_codes(fn)) - collections.Counter(a['category_id'] for a in train_records[fn]['annotations']))[code]
           for fn, code in MISSING_REGIONS), '영역의 K-코드가 그 이미지에서 빠진 K-코드가 아니다'
class_sizes = {code: np.median([sorted(bbox[2:], reverse=True) for c, bbox, _ in reference if c == code], axis=0).astype(int)
               for code in set(reference_codes)}  # 클래스 → 라벨 box의 긴 변, 짧은 변 중앙값 (알약이 놓인 방향과 상관없이 비교)
drawn_boxes = {(fn, code): drawn_box(pill_extent(Image.open(TRAIN_IMG_DIR / fn), region), class_margins[code])
               for (fn, code), region in MISSING_REGIONS.items()}  # (이미지, K-코드) → 그려 넣은 xywh box

fig, axes = plt.subplots(2, 4, figsize=(24, 18), layout='constrained')
for ax, ((fn, code), box) in zip(axes.flat, drawn_boxes.items()):
    ax.imshow(Image.open(TRAIN_IMG_DIR / fn))
    for ann in train_records[fn]['annotations']:
        x, y, w, h = ann['bbox']
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=2.5))
        ax.text(x, y - 8, pill_label(ann), fontsize=9, va='bottom', bbox=dict(facecolor='lime', alpha=0.85, pad=1.5, linewidth=0))
    x0, y0, x1, y1 = MISSING_REGIONS[(fn, code)]
    ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor='white', linewidth=1.5, linestyle=':'))
    x, y, w, h = box
    ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='deepskyblue', linewidth=2.5, linestyle='--'))
    ax.text(x, y - 8, pill_label(class_example[code]), fontsize=9, va='bottom', bbox=dict(facecolor='deepskyblue', alpha=0.85, pad=1.5, linewidth=0))
    ax.set_title(f'{fn}\n그려 넣은 box {box} (긴 변×짧은 변 {max(box[2:])}×{min(box[2:])})\n같은 클래스 라벨: 긴 변×짧은 변 중앙값 {class_sizes[code][0]}×{class_sizes[code][1]}, '
                 f'검증 IoU 중앙값 {np.median(ref_ious[reference_codes == code]):.3f}', fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle('라벨이 누락된 8장에 그려 넣은 box (파란 점선 = 그려 넣은 box, 흰 점선 = 눈으로 정한 영역, 연두 = 원래 라벨, 글자 = K-코드(각인))', fontsize=14)
save_fig(fig, '2-1d_drawn_boxes')
plt.show()

**인사이트**
- 그려 넣은 box 8개 모두 빠진 알약을 감싸고, 다른 알약이나 그림자를 넣지 않는다.
    - 그린 box는 그 사진의 다른 box와 겹치지 않는다.
- 그린 box의 크기(긴 변×짧은 변)는 같은 클래스 라벨 box의 중앙값과 비슷하다.
    - HT 정제(K-003351) 4개: 194~195×188~191 (라벨 중앙값 188×185)
    - K-020014 348×340 (348×339), K-032310 377×250 (368×262), K-033880 438×315 (437×303), K-004543 207×203 (208×204)

**결론**
- 8장은 그린 box를 넣어 학습에 쓴다. 그린 box(`drawn_boxes`)는 2-5의 정제 라벨에 넣는다.

### 2-2. 좌표 오류
- 이미지(976×1280) 밖으로 나간 bbox를 찾는다.
- annotation의 area가 bbox 너비×높이와 다른 경우를 찾는다.
    - area는 COCO 평가에서 객체 크기 구간을 나눌 때만 쓰고, YOLO 학습에는 쓰지 않는다.
- 같은 이미지 안에서 다른 클래스의 box와 좌표가 똑같은 box를 찾는다.
    - 서로 다른 알약 두 개가 같은 자리에 있을 수는 없으므로, 한 알약의 box를 다른 클래스에도 복사한 오류이다.

In [ ]:
coord_errors = []  # 표의 행마다 (이미지 파일명, annotation, 문제 목록)
coord_rows = []
for fn, record in sorted(train_records.items()):
    anns = record['annotations']
    for ann in anns:
        x, y, w, h = ann['bbox']
        problems = []
        if x < 0 or y < 0 or x + w > IMG_W or y + h > IMG_H:
            problems.append('이미지 밖')
        if ann['area'] != w * h:
            problems.append('area 불일치')
        twins = [f"K-{other['category_id']:06d}" for other in anns if other is not ann and other['bbox'] == ann['bbox']]
        if twins:
            problems.append(f"{', '.join(twins)}와 box가 똑같음")
        if problems:
            coord_errors.append((fn, ann, problems))
            coord_rows.append([f'`{fn}`', f"K-{ann['category_id']:06d}", ann['bbox'], ann['area'], w * h, ', '.join(problems)])
show_table(['이미지', 'K-코드', 'bbox [x, y, w, h]', 'area', '너비×높이', '문제'], coord_rows)

**인사이트**
- 이미지 밖으로 나간 bbox는 1개이다. K-018357의 x가 6567로, 이미지 너비 976을 넘는다.
- 3장에서 서로 다른 두 클래스의 box 좌표가 똑같다.
    - 3장은 아래 그림에서 사진으로 확인한다.
    - K-016548과 K-033009, K-003351과 K-020238, K-003351과 K-031863이다.
    - 한 알약의 box를 다른 클래스에도 복사한 것이라, 한 클래스는 box가 틀리고 다른 알약은 box가 없는 셈이다.
- area가 너비×높이와 다른 annotation은 6개이다. 차이는 60~23,200이다.

**결론**
- x 좌표 오타는 아래 그림에서 고칠 값을 정한다.
- box가 똑같은 3장은 사진에서 box가 감싼 알약을 확인하고, 클래스가 맞지 않는 쪽의 box를 box가 없는 알약으로 옮긴다.
- area 불일치 6개는 box를 확대해 보고, area와 bbox 중 어느 쪽이 틀렸는지 정한다.

- 이미지 밖으로 나간 bbox는 x 좌표에 숫자 하나가 더 들어간 오타로 보인다.
- x 값에서 숫자 하나를 뺀 후보마다 box를 그려 보고, box가 알약을 감싸는 후보를 눈으로 고른다.

In [ ]:
oob_fn, oob_ann = next((fn, a) for fn, r in sorted(train_records.items()) for a in r['annotations'] if a['bbox'][0] > IMG_W)
x_text = str(oob_ann['bbox'][0])
x_candidates = sorted({int(x_text[:i] + x_text[i + 1:]) for i in range(len(x_text))})
oob_image = Image.open(TRAIN_IMG_DIR / oob_fn)

fig, axes = plt.subplots(1, len(x_candidates), figsize=(4 * len(x_candidates), 4.5), layout='constrained')
for ax, x in zip(axes, x_candidates):
    box = [x, *oob_ann['bbox'][1:]]
    ax.imshow(draw_boxes(oob_image, [box], width=6))
    ax.set_xlim(x - 150, x + box[2] + 150)
    ax.set_ylim(box[1] + box[3] + 150, box[1] - 150)
    ax.set_title(f'x 후보 {x}')
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(f'이미지 밖 bbox의 x 후보 (원래 x = {x_text}, K-{oob_ann["category_id"]:06d}, 빨간 box가 알약을 감싸는 후보를 고른다)')
save_fig(fig, '2-2_oob_candidates')
plt.show()

**인사이트**
- x = 567일 때 box가 알약(GLT400 연질 캡슐)을 감싼다.
- 나머지 후보(656, 657, 667)는 box가 알약 오른쪽으로 89~100픽셀 밀려나, 알약 왼쪽이 box 밖으로 나간다.

**결론**
- K-018357의 x를 6567에서 567로 고쳐서 쓴다.

- 2-2 표에서 두 클래스의 box가 똑같은 3장에 라벨 box를 모두 겹쳐 그린다.
    - 빨간 box는 두 클래스가 함께 쓰는 box이고, 그 위에 두 클래스의 글자를 위아래로 쌓는다. 연두 box는 나머지 라벨이다.
    - box 위 글자는 K-코드와 각인이다. (2-1의 그림과 같음)
    - #번호는 2-2 표의 행 순서(위에서부터 #1~#13)이다.
    - 그림 제목 둘째 줄은 똑같은 box의 좌표와 두 box의 IoU(겹친 넓이 ÷ 합친 넓이, 0이면 전혀 겹치지 않고 1이면 완전히 같음)이다.
- 똑같은 box가 두 클래스 중 어느 알약을 감싸는지, box가 없는 알약이 사진에 있는지 본다.

In [ ]:
coord_numbers = {id(ann): n for n, (_, ann, _) in enumerate(coord_errors, start=1)}  # annotation → 2-2 표의 행 번호
twin_images = list(dict.fromkeys(fn for fn, _, problems in coord_errors if any('box가 똑같음' in p for p in problems)))

fig, axes = plt.subplots(1, len(twin_images), figsize=(21, 10.5), layout='constrained')
for ax, fn in zip(axes, twin_images):
    ax.imshow(Image.open(TRAIN_IMG_DIR / fn))
    anns = train_records[fn]['annotations']
    stacked = collections.Counter()  # 좌표가 똑같은 box는 글자를 위로 쌓는다
    for ann in anns:
        x, y, w, h = ann['bbox']
        n = coord_numbers.get(id(ann))
        color = 'red' if n else 'lime'
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2.5))
        ax.text(x, y - 8 - 40 * stacked[(x, y, w, h)], f'#{n} {pill_label(ann)}' if n else pill_label(ann),
                fontsize=11, va='bottom', bbox=dict(facecolor=color, alpha=0.85, pad=1.5, linewidth=0))
        stacked[(x, y, w, h)] += 1
    twin_a, twin_b = [ann for ann in anns if id(ann) in coord_numbers]
    ax.set_title(f"{fn}\n#{coord_numbers[id(twin_a)]}·#{coord_numbers[id(twin_b)]} box {twin_a['bbox']}, "
                 f"IoU {box_iou(twin_a['bbox'], twin_b['bbox']):.2f}", fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(f'두 클래스의 box가 똑같은 이미지 {len(twin_images)}장 (빨간 box = 두 클래스가 함께 쓰는 box, 연두 box = 나머지 라벨, #번호 = 2-2 표의 행 순서)',
             fontsize=14)
save_fig(fig, '2-2_duplicate_boxes')
plt.show()

**인사이트**
- 3장 모두 두 클래스의 box가 IoU 1.00으로 완전히 겹친다.
    - 똑같은 box 쌍 말고는 3장 안에서 겹치는 box가 없다.
- 똑같은 box는 두 클래스 중 한 알약만 감싸고, 다른 알약 하나에는 box가 없다.
    - #1·#2: box는 S300 정제(K-033009)를 감싼다. 왼쪽 아래 노란 원형 GB 정제에는 box가 없다.
        - K-016548은 메타데이터에 각인이 비어 있지만, 나머지 세 알약(S300, FZ12, BSP)이 각자의 box와 각인이 맞으므로 box가 없는 GB 정제가 K-016548이다.
    - #7·#8: box는 HT 정제(K-003351)를 감싼다. 오른쪽 위 분홍색 원형 정제(각인 75, K-020238)에는 box가 없다.
    - #10·#11: box는 GIL 정제(K-031863)를 감싼다. 아래쪽 HT 정제(K-003351)에는 box가 없다.

**결론**
- 똑같은 box 중 클래스가 맞지 않는 쪽(K-016548, K-020238, K-003351)의 box를 box가 없는 알약으로 옮긴다. 옮길 자리는 아래 그림에서 정한다.
    - 틀린 클래스 하나를 지우기만 하면 box가 없는 알약이 남는다. 그대로 학습하면 모델이 그 알약을 배경으로 배운다.

- 두 클래스의 box가 똑같은 3장에서, 똑같은 box 중 클래스가 맞지 않는 쪽을 box가 없는 알약으로 옮긴다.
    - 옮길 클래스는 위 그림에서 찾은 K-016548(GB), K-020238(75), K-003351(HT)이다.
    - box가 없는 알약 하나만 들어가는 영역을 사진을 보고 정하고, 2-1의 방법(알약 범위 + 같은 클래스 라벨 여백 중앙값)으로 box를 다시 그린다.
    - 칸 제목 셋째 줄은 같은 클래스 라벨 box의 긴 변×짧은 변 중앙값과, 2-1 검증에서 그 클래스의 IoU 중앙값이다.
- 옮긴 box가 알약을 감싸는지 본다.

In [ ]:
MOVED_REGIONS = {  # (이미지, box를 옮길 K-코드) → box가 없는 알약 하나만 들어가는 영역 [x0, y0, x1, y1] (위 그림에서 확인)
    ('K-001900-016548-019607-033009_0_2_0_2_70_000_200.png', 16548): [70, 830, 360, 1120],
    ('K-003351-020238-031863_0_2_0_2_70_000_200.png', 20238): [570, 270, 830, 530],
    ('K-003351-029667-031863_0_2_0_2_70_000_200.png', 3351): [350, 830, 580, 1060],
}
assert [fn for fn, _ in MOVED_REGIONS] == twin_images
moved_boxes = {(fn, code): drawn_box(pill_extent(Image.open(TRAIN_IMG_DIR / fn), region), class_margins[code])
               for (fn, code), region in MOVED_REGIONS.items()}  # (이미지, K-코드) → 옮긴 xywh box

fig, axes = plt.subplots(1, len(moved_boxes), figsize=(21, 10.5), layout='constrained')
for ax, ((fn, code), box) in zip(axes, moved_boxes.items()):
    ax.imshow(Image.open(TRAIN_IMG_DIR / fn))
    for ann in train_records[fn]['annotations']:
        x, y, w, h = ann['bbox']
        moved = ann['category_id'] == code
        if moved:
            x, y, w, h = box
        color = 'deepskyblue' if moved else 'lime'
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2.5, linestyle='--' if moved else '-'))
        ax.text(x, y - 8, pill_label(ann), fontsize=11, va='bottom', bbox=dict(facecolor=color, alpha=0.85, pad=1.5, linewidth=0))
    ax.set_title(f'{fn}\nK-{code:06d}의 box를 {box}로 옮김 (긴 변×짧은 변 {max(box[2:])}×{min(box[2:])})\n같은 클래스 라벨: 긴 변×짧은 변 중앙값 {class_sizes[code][0]}×{class_sizes[code][1]}, '
                 f'검증 IoU 중앙값 {np.median(ref_ious[reference_codes == code]):.3f}', fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle('두 클래스의 box가 똑같은 3장에서 box 하나를 옮긴 결과 (파란 점선 = box가 없던 알약으로 옮긴 box, 연두 = 그대로 둔 라벨)', fontsize=14)
save_fig(fig, '2-2_moved_boxes')
plt.show()

**인사이트**
- 옮긴 box 3개 모두 box가 없던 알약을 감싼다.
    - K-020238(75) 223×220 (라벨 중앙값 228×224), K-003351(HT) 193×188 (188×185)
    - K-016548(GB)은 245×239로, 라벨 중앙값(238×227)보다 7~12픽셀 크다.
- 그대로 둔 box(K-033009, K-003351, K-031863)는 자기 알약을 감싼다. (위 그림)
- 옮긴 뒤에는 3장 모두 알약마다 box가 하나씩 있고, 서로 겹치는 box가 없다.

**결론**
- 3장은 box 하나를 옮겨 학습에 쓴다. 옮긴 box(`moved_boxes`)는 2-5의 정제 라벨에 적용한다.

- 2-2 표의 13행에 표 순서대로 #1~#13 번호를 붙이고, 그 행이 있는 이미지 9장에 라벨 box를 모두 겹쳐 그린다.
    - 빨간 box는 표에 나온 box이고, 연두 box는 문제가 없는 box이다.
    - box 위 글자는 K-코드와 메타데이터의 각인(print_front/print_back)이다. 이 글자를 box 안 알약에 찍힌 글자와 대조하면, box가 어느 알약을 감싸는지 알 수 있다.
- box가 똑같은 3장(#1·#2, #7·#8, #10·#11)은 위 그림에서 봤다. 나머지 6장에서 이미지 밖 box와 area 불일치 box가 알약을 감싸는지 본다.
- 이미지 밖 box(#4)는 그릴 수 없어서 그림 제목에 적는다.

In [ ]:
coord_images = list(dict.fromkeys(fn for fn, _, _ in coord_errors))

fig, axes = plt.subplots(3, 3, figsize=(18, 23), layout='constrained')
for ax, fn in zip(axes.flat, coord_images):
    ax.imshow(Image.open(TRAIN_IMG_DIR / fn))
    stacked = collections.Counter()  # 좌표가 똑같은 box는 글자를 위로 쌓는다
    notes = []
    for ann in train_records[fn]['annotations']:
        n = coord_numbers.get(id(ann))
        x, y, w, h = ann['bbox']
        text = f'#{n} {pill_label(ann)}' if n else pill_label(ann)
        if x >= IMG_W:
            notes.append(f'{text}: x = {x}, 이미지 밖이라 그리지 않음')
            continue
        color = 'red' if n else 'lime'
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2.5))
        ax.text(x, y - 8 - 34 * stacked[(x, y, w, h)], text, fontsize=9, va='bottom',
                bbox=dict(facecolor=color, alpha=0.85, pad=1.5, linewidth=0))
        stacked[(x, y, w, h)] += 1
    numbers = ', '.join(f'#{coord_numbers[id(ann)]}' for f, ann, _ in coord_errors if f == fn)
    ax.set_title('\n'.join([numbers, fn, *notes]), fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle('2-2 표의 13행이 있는 이미지 9장 (빨간 box = 표의 행 #1~#13, 연두 box = 문제없는 라벨, 글자 = K-코드(각인))', fontsize=14)
save_fig(fig, '2-2c_coord_errors')
plt.show()

**인사이트**
- #4는 box가 이미지 밖에 있어서, 오른쪽 GLT400 캡슐에 box가 없는 상태이다.
- area 불일치 6개는 이 크기에서 대부분 box가 알약을 감싼다. #5만 HM-CN 알약의 아래쪽이 box 밖으로 크게 나간다.

**결론**
- area 불일치 6개는 아래 그림에서 box를 확대해, 고칠 값을 정한다.

- area 불일치 6개는 area 값이 틀렸을 수도 있고, bbox의 너비나 높이가 틀렸을 수도 있다.
    - area가 bbox 너비로 나눠떨어지면 몫을 높이 후보로, 높이로 나눠떨어지면 몫을 너비 후보로 둔다. 원본 area와 맞는 box 크기가 하나 더 있다는 뜻이다.
- box마다 확대해 라벨 bbox(빨간 실선)와 후보 box(파란 점선)를 겹쳐 그린다.
    - 알약 둘레의 여백이 네 변에서 비슷한 쪽을 맞는 크기로 본다. 1-3에서 본 train box는 알약을 빈틈없이 감쌌다.
    - 축 눈금은 원본 픽셀 좌표이다.

In [ ]:
area_errors = [(n, fn, ann) for n, (fn, ann, problems) in enumerate(coord_errors, start=1) if 'area 불일치' in problems]

fig, axes = plt.subplots(2, 3, figsize=(18, 14), layout='constrained')
for ax, (n, fn, ann) in zip(axes.flat, area_errors):
    x, y, w, h = ann['bbox']
    # area가 너비나 높이로 나눠떨어지면, 원본 area와 맞는 box 크기 후보를 만든다
    if ann['area'] % w == 0:
        candidate, candidate_text = [x, y, w, ann['area'] // w], f"높이 {h} → {ann['area'] // w}"
    elif ann['area'] % h == 0:
        candidate, candidate_text = [x, y, ann['area'] // h, h], f"너비 {w} → {ann['area'] // h}"
    else:
        candidate, candidate_text = None, '없음 (너비, 높이 어느 쪽으로도 나눠떨어지지 않음)'
    ax.imshow(Image.open(TRAIN_IMG_DIR / fn))
    ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='red', linewidth=2))
    right, bottom = x + w, y + h
    if candidate:
        ax.add_patch(plt.Rectangle((x, y), candidate[2], candidate[3], fill=False, edgecolor='deepskyblue', linewidth=2, linestyle='--'))
        right, bottom = max(right, x + candidate[2]), max(bottom, y + candidate[3])
    ax.set_xlim(x - 30, right + 30)
    ax.set_ylim(bottom + 30, y - 30)
    ax.set_title(f"#{n} {pill_label(ann)}\nbbox 너비×높이 = {w}×{h} = {w * h:,}, 원본 area = {ann['area']:,}\n"
                 f"area에 맞는 크기 후보: {candidate_text}", fontsize=10)
fig.suptitle('area 불일치 6개 확대 (빨간 실선 = 라벨 bbox, 파란 점선 = 원본 area에 맞춘 box 후보, 축 = 원본 픽셀 좌표)', fontsize=14)
save_fig(fig, '2-2d_area_zoom')
plt.show()

**인사이트**
- 6개 중 3개(#5, #6, #9)는 원본 area가 bbox 너비나 높이로 나눠떨어진다.
    - 몫은 원래 값과 숫자 하나만 다르다. (높이 241 → 341, 너비 322 → 332, 높이 182 → 192)
- #5는 bbox 높이의 오타이다.
    - 높이 241이면 box 아래 변(y 442)이 알약 가운데를 지나고, 알약 아래쪽 약 80픽셀이 box 밖으로 나간다.
    - 높이 341이면 위아래 여백이 약 18픽셀로 같다.
    - 원래 box와 고친 box의 IoU는 0.71(241÷341)이다. 평가 지표의 가장 낮은 기준인 IoU 0.75에도 못 미친다.
- #9도 bbox 높이의 오타로 본다.
    - 높이 182이면 box 아래 변이 알약 가장자리에 닿는다.
    - 높이 192이면 아래 여백이 약 9픽셀로, 위·왼쪽·오른쪽 여백(10~12픽셀)과 비슷하다.
    - 원래 box와 고친 box의 IoU는 0.95(182÷192)라 영향은 작다.
- #6은 area 값의 오타로 본다.
    - 너비 322이면 왼쪽과 오른쪽 여백이 약 8픽셀과 7픽셀로 비슷하다. 332이면 오른쪽 여백만 약 17픽셀이 된다.
- 나머지 3개(#3, #12, #13)는 area가 나눠떨어지지 않고, box가 알약을 감싼다.
    - 원본 area가 너비×높이와 숫자 하나만 다르다. (161,036 → 161,096, 129,780 → 129,480, 65,009 → 62,009)

**결론**
- #5(K-019232)의 bbox 높이를 241에서 341로, #9(K-003351)의 bbox 높이를 182에서 192로 고친다.
- 나머지 4개는 bbox를 그대로 쓴다.
- area는 bbox를 고친 뒤 너비×높이로 다시 계산하고, 원본 area 값은 쓰지 않는다.
    - 높이를 고친 2개는 다시 계산한 값이 원본 area와 같다.

### 2-3. 클래스별 crop 갤러리
- train의 모든 bbox를 클래스별로 잘라 한 줄에 이어 붙인다. 이미지 밖 bbox 1개(2-2)는 뺀다.
    - 여백 없이 bbox 그대로 자르고, 높이만 맞춰 원래 비율대로 줄인다. 헐거운 box는 배경이 넓게 보이고, 너무 작은 box는 알약 가장자리가 잘린다.
    - 한 클래스 줄 안에 다른 알약이 섞여 있으면 클래스가 잘못 붙은 라벨이다.
- 줄 앞의 글자는 K-코드와 객체 수이다. 객체가 많은 클래스부터 놓고, 한 클래스 안의 crop은 파일명 순서이다.

In [ ]:
TILE_HEIGHT = 72  # crop 하나의 높이 (픽셀)
ROW_WIDTH = 1800  # crop을 이어 붙이는 한 줄의 최대 너비 (픽셀)
LABEL_WIDTH = 230  # 줄 앞 클래스 이름 칸의 너비 (픽셀)
GAP = 4  # crop 사이 간격 (픽셀)

tiles_by_class = collections.defaultdict(list)  # K-코드 → [(파일명, crop)]
for fn in train_files:
    image = Image.open(TRAIN_IMG_DIR / fn).convert('RGB')
    for ann in train_records[fn]['annotations']:
        x, y, w, h = ann['bbox']
        if x >= IMG_W:  # 이미지 밖 bbox(2-2)는 뺀다
            continue
        crop = image.crop((x, y, x + w, y + h)).resize((max(1, round(w * TILE_HEIGHT / h)), TILE_HEIGHT))
        tiles_by_class[ann['category_id']].append((fn, crop))
class_order = sorted(tiles_by_class, key=lambda c: (-len(tiles_by_class[c]), c))

# 클래스마다 crop을 한 줄에 이어 붙이고, 줄이 ROW_WIDTH를 넘으면 다음 줄로 넘긴다
blocks = []  # (K-코드, [줄마다 crop 목록])
for code in class_order:
    class_lines, line, width = [], [], 0
    for _, crop in tiles_by_class[code]:
        if line and width + crop.width > ROW_WIDTH:
            class_lines.append(line)
            line, width = [], 0
        line.append(crop)
        width += crop.width + GAP
    class_lines.append(line)
    blocks.append((code, class_lines))

n_lines = sum(len(class_lines) for _, class_lines in blocks)
canvas = Image.new('RGB', (LABEL_WIDTH + ROW_WIDTH, n_lines * (TILE_HEIGHT + GAP) + len(blocks) * 3 * GAP), 'white')
draw = ImageDraw.Draw(canvas)
font = ImageFont.load_default(size=26)
y = 0
for code, class_lines in blocks:
    draw.text((8, y + TILE_HEIGHT // 2 - 14), f'K-{code:06d} ({len(tiles_by_class[code])})', fill='black', font=font)
    for line in class_lines:
        x = LABEL_WIDTH
        for crop in line:
            canvas.paste(crop, (x, y))
            x += crop.width + GAP
        y += TILE_HEIGHT + GAP
    draw.line([(0, y + GAP), (canvas.width, y + GAP)], fill='gray', width=2)  # 클래스 사이 구분선
    y += 3 * GAP

fig, ax = plt.subplots(figsize=(canvas.width / 100, canvas.height / 100 + 0.6), layout='constrained')
ax.imshow(canvas)
ax.axis('off')
ax.set_title(f'클래스별 bbox crop 전체 {sum(len(t) for t in tiles_by_class.values())}개 '
             f'(줄 앞 = K-코드와 객체 수, 객체가 많은 클래스부터, 한 클래스 안은 파일명 순서)', fontsize=16)
save_fig(fig, '2-3_class_crops')
plt.show()

**인사이트**
- 762개 crop 중 6개가 그 줄의 다른 crop과 다른 알약이다. 모두 2-2의 box가 똑같은 3장과 아래 2장에서 나왔다.
    - box가 똑같은 3장(2-2): K-016548 줄의 S300 알약, K-020238 줄의 HT 알약, K-003351 줄의 흰색 GIL 알약
    - `K-003483-025367-027733-035206_..._75_...`: K-035206 줄에 A1 알약, K-027733 줄에 337 알약이 있다. 두 클래스의 라벨이 서로 바뀌었다.
        - 두 코드 모두 파일명에 있어서 2-1의 대조로는 잡히지 않았다.
    - `K-003351-020014-020238_..._90_...`: K-003351 줄에 알약 없이 바닥만 찍힌 crop이 있다. box가 알약이 없는 곳을 감싼다.
        - 사진은 아래 그림에서 확인한다.
- 나머지 crop은 줄마다 같은 알약이고, box가 알약 가장자리를 조금 여유 있게 감싼다.
- 같은 클래스 안에서도 crop 크기와 방향이 다르다. (예: K-041768 CJ 캡슐)
    - 촬영 각도(70°, 75°, 90°)와 알약이 놓인 방향이 달라서 생기는 차이로 보인다.

**결론**
- 라벨이 서로 바뀐 1장은 두 라벨을 서로 바꾼다.
- box 안에 알약이 없는 1장은 아래 그림에서 고칠 값을 정한다.
- box 크기는 알약을 일정한 여유로 감싸므로, box를 일괄로 늘리거나 줄이는 보정은 하지 않는다.
- 사진의 각도를 180도 틀어서 보는 증강은 고려해볼 만 하다.  

- 갤러리의 K-003351 줄에서 바닥만 찍힌 crop이 나온 이미지 1장에 라벨 box를 모두 겹쳐 그린다.
    - 왼쪽은 이미지 전체이고, 오른쪽은 바닥을 감싼 box 부근을 확대한 것이다. 오른쪽 축 눈금은 원본 픽셀 좌표이다.
    - 빨간 실선은 바닥을 감싼 K-003351 box이고, 연두 box는 나머지 라벨이다. box 위 글자는 K-코드와 각인이다. (2-1의 그림과 같음)
- K-003351 알약(HT 정제)이 사진에 있는지, box가 어느 쪽으로 어긋났는지 본다.
    - 2-2의 x 좌표 오타처럼 숫자 하나가 틀렸을 수 있다. x의 숫자 하나를 바꿔 box가 알약을 감싸는 값을 찾고, 그 box를 파란 점선으로 함께 그린다.

In [ ]:
SHIFTED_FILE, SHIFTED_CODE = 'K-003351-020014-020238_0_2_0_2_90_000_200.png', 3351  # 갤러리의 K-003351 줄에서 바닥만 찍힌 crop이 나온 box
SHIFTED_X = 385  # 라벨 x 185에서 숫자 하나를 바꾼 값. box가 옆의 HT 정제를 감싼다

shifted_ann = next(a for a in train_records[SHIFTED_FILE]['annotations'] if a['category_id'] == SHIFTED_CODE)
x, y, w, h = shifted_ann['bbox']
fig, axes = plt.subplots(1, 2, figsize=(16, 9.5), layout='constrained', gridspec_kw={'width_ratios': [1, 1.3]})
for ax in axes:
    ax.imshow(Image.open(TRAIN_IMG_DIR / SHIFTED_FILE))
    for ann in train_records[SHIFTED_FILE]['annotations']:
        color = 'red' if ann is shifted_ann else 'lime'
        bx, by, bw, bh = ann['bbox']
        ax.add_patch(plt.Rectangle((bx, by), bw, bh, fill=False, edgecolor=color, linewidth=2.5))
        ax.text(bx, by - 8, pill_label(ann), fontsize=10, va='bottom', clip_on=True,
                bbox=dict(facecolor=color, alpha=0.85, pad=1.5, linewidth=0))
    ax.add_patch(plt.Rectangle((SHIFTED_X, y), w, h, fill=False, edgecolor='deepskyblue', linewidth=2.5, linestyle='--'))
    ax.text(SHIFTED_X, y + h + 8, f'x {SHIFTED_X}로 옮긴 box', fontsize=10, va='top', clip_on=True,
            bbox=dict(facecolor='deepskyblue', alpha=0.85, pad=1.5, linewidth=0))
axes[0].set_title(SHIFTED_FILE, fontsize=11)
axes[0].set_xticks([])
axes[0].set_yticks([])
axes[1].set_xlim(x - 40, SHIFTED_X + w + 40)
axes[1].set_ylim(y + h + 60, y - 60)
axes[1].set_title(f'두 box 확대 (라벨 bbox {shifted_ann["bbox"]}, 축 = 원본 픽셀 좌표)', fontsize=11)
fig.suptitle(f'바닥을 감싼 K-{SHIFTED_CODE:06d} box (빨간 실선 = 라벨 box, 파란 점선 = x만 {x}에서 {SHIFTED_X}로 옮긴 box, '
             f'연두 = 나머지 라벨, 글자 = K-코드(각인))', fontsize=13)
save_fig(fig, '2-3b_shifted_box')
plt.show()

**인사이트**
- 바닥을 감싼 box 바로 오른쪽에 HT 정제(K-003351)가 있다. 사진에서 box가 없는 알약은 이 HT 정제 하나이다.
- box는 HT 정제에서 가로로만 약 200픽셀 어긋났다.
    - 세로는 맞다. 라벨 box의 위아래(y 249~432)가 알약의 위아래를 감싼다.
    - x를 185에서 385로 바꾸면(백의 자리 숫자 하나) box가 HT 정제를 감싼다. 좌우 여백이 약 11픽셀로 같고, 위 여백은 약 6픽셀, 아래 여백은 약 11픽셀이다.
- 라벨 box와 옮긴 box는 전혀 겹치지 않는다(IoU 0). 고치지 않고 학습하면 HT 정제는 배경으로, 빈 바닥은 HT 정제로 배운다.
- 나머지 두 box(K-020014, K-020238)는 각인이 맞고 알약을 감싼다.

**결론**
- K-003351의 x를 185에서 385로 고쳐서 쓴다. 2-2의 x 좌표 오타(6567 → 567)와 같이 숫자 하나가 틀린 오타로 본다.

### 2-4. train-test 중복
- 파일 내용의 해시(MD5, 파일 내용이 1바이트라도 다르면 값이 달라지는 지문)가 같은 이미지 쌍을 센다.
    - 해시가 같으면 완전히 같은 파일이다.

In [ ]:
def file_hashes(image_dir, files):
    """파일마다 MD5 해시를 구한다."""
    return {fn: hashlib.md5((image_dir / fn).read_bytes()).hexdigest() for fn in files}


train_hash = file_hashes(TRAIN_IMG_DIR, train_files)
test_hash = file_hashes(TEST_IMG_DIR, test_files)


def duplicate_pairs(hashes):
    """해시가 같은 파일 쌍의 수를 센다."""
    return sum(n * (n - 1) // 2 for n in collections.Counter(hashes).values())


cross = sum(1 for h in test_hash.values() if h in set(train_hash.values()))
show_table(['비교', '같은 파일 쌍 수'], [
    ['train 내부', duplicate_pairs(train_hash.values())],
    ['test 내부', duplicate_pairs(test_hash.values())],
    ['train과 test 사이', cross],
])

**인사이트**
- 파일 내용이 완전히 같은 이미지는 train 내부, test 내부, train과 test 사이 모두 없다.

**결론**
- 완전 중복으로 지울 이미지는 없다.
- 같은 조합을 다른 각도에서 찍은 비슷한 사진은 파일이 달라 여기서 잡히지 않는다.

### 2-5. 라벨 정제 목록
- 2-1~2-3에서 찾은 라벨 오류를 이미지별로 모으고, 아래 규칙으로 처리 방법을 정한다.
    - 라벨 누락(2-1): 사진에서 그린 box를 넣는다.
    - 두 클래스의 box가 똑같음(2-2): 클래스가 맞지 않는 쪽의 box를 box가 없는 알약으로 옮긴다.
    - x 좌표 오타(2-2, 2-3): 후보 중 box가 알약을 감싸는 값으로 고친다.
    - bbox 높이 오타(2-2): 원본 area에 맞춘 높이로 고친다.
    - area 값 오타(2-2): bbox는 그대로 두고 area를 너비×높이로 다시 계산한다.
    - 라벨이 서로 바뀜(2-3): 두 라벨을 서로 바꾼다.
- 2-2와 2-3의 그림에서 눈으로 찾은 오류는 코드에 상수로 적는다.
- 이 목록을 적용한 정제 라벨(`clean_records`)을 만든다.
    - 고칠 값을 적용하고, 그린 box를 넣고, 모든 area를 너비×높이로 다시 계산한다.
    - 그린 box의 알약 메타데이터(제품명, 각인 등)는 같은 클래스의 다른 annotation에서 가져오고, 이미지 필드(파일명, 촬영 각도 등)는 그 이미지의 값으로 바꾼다.
    - 라벨을 서로 바꿀 때는 클래스와 함께 알약 메타데이터(제품명, 각인 등)도 바꾼다.

In [ ]:
# 그림에서 눈으로 확인한 라벨 오류 (2-2, 2-3의 인사이트 참고)
FIXED_X = 567  # 이미지 밖 bbox의 x 후보 중 box가 알약을 감싸는 값 (2-2)
SWAPPED_LABELS = {  # 이미지 → 라벨이 서로 바뀐 두 K-코드 (2-3)
    'K-003483-025367-027733-035206_0_2_0_2_75_000_200.png': (27733, 35206),
}
FIXED_HEIGHTS = {  # (이미지, K-코드) → 원본 area에 맞춰 고친 bbox 높이 (2-2)
    ('K-003351-019232-029667_0_2_1_2_70_000_200.png', 19232): 341,
    ('K-003351-021325-033880_0_2_0_2_70_000_200.png', 3351): 192,
}
SHIFTED_BOXES = {(SHIFTED_FILE, SHIFTED_CODE): SHIFTED_X}  # (이미지, K-코드) → 옆 알약 대신 바닥을 감싼 box의 고친 x (2-3)
assert FIXED_X in x_candidates and set(SWAPPED_LABELS) <= set(train_files)
IMAGE_FIELDS = ['back_color', 'camera_la', 'camera_lo', 'drug_S', 'drug_dir', 'file_name', 'height', 'id', 'imgfile',
                'light_color', 'size', 'width']  # 메타데이터 중 같은 이미지의 알약끼리 값이 같은 필드 (나머지는 알약 필드)

problems_by_image = collections.defaultdict(list)  # 이미지 파일명 → [(문제, 처리)]
for (fn, code), box in drawn_boxes.items():
    problems_by_image[fn].append((f'K-{code:06d} 라벨 누락 (2-1)', f'box {box}를 그려 넣음'))
for fn, record in sorted(train_records.items()):
    anns = record['annotations']
    for ann in anns:
        if ann['bbox'][0] >= IMG_W:
            problems_by_image[fn].append((f"K-{ann['category_id']:06d}의 x 좌표 오타 {ann['bbox'][0]} (2-2)", f'x를 {FIXED_X}로 고침'))
        # area가 너비×높이와 다른 box 중 높이를 고치는 것(FIXED_HEIGHTS)을 뺀 나머지는 area 값의 오타이다
        if ann['area'] != ann['bbox'][2] * ann['bbox'][3] and (fn, ann['category_id']) not in FIXED_HEIGHTS:
            problems_by_image[fn].append((f"K-{ann['category_id']:06d}의 area 오타 (2-2)", 'area를 너비×높이로 다시 계산'))
for (fn, code), box in moved_boxes.items():
    problems_by_image[fn].append((f'K-{code:06d}의 box가 다른 클래스와 똑같음 (2-2)', f'box를 {box}로 옮김'))
for (fn, code), height in FIXED_HEIGHTS.items():
    ann = next(a for a in train_records[fn]['annotations'] if a['category_id'] == code)
    assert ann['area'] == ann['bbox'][2] * height, '고친 높이로 계산한 넓이가 원본 area와 다르다'
    problems_by_image[fn].append((f"K-{code:06d}의 bbox 높이 오타 {ann['bbox'][3]} (2-2)", f'높이를 {height}로 고침'))
for fn, (a, b) in SWAPPED_LABELS.items():
    problems_by_image[fn].append((f'K-{a:06d}과 K-{b:06d}의 라벨이 서로 바뀜 (2-3)', '두 라벨을 서로 바꿈'))
for (fn, code), new_x in SHIFTED_BOXES.items():
    ann = next(a for a in train_records[fn]['annotations'] if a['category_id'] == code)
    problems_by_image[fn].append((f"K-{code:06d}의 x 좌표 오타 {ann['bbox'][0]} (2-3)", f'x를 {new_x}로 고침'))

# 정제 라벨: 원본 train_records는 그대로 두고, 고친 라벨을 새로 만든다
clean_records = {}
for fn, record in sorted(train_records.items()):
    by_code = {ann['category_id']: ann for ann in record['annotations']}
    anns = []
    for ann in record['annotations']:
        x, y, w, h = moved_boxes.get((fn, ann['category_id']), ann['bbox'])
        x = FIXED_X if x >= IMG_W else SHIFTED_BOXES.get((fn, ann['category_id']), x)
        h = FIXED_HEIGHTS.get((fn, ann['category_id']), h)
        label = ann
        if ann['category_id'] in SWAPPED_LABELS.get(fn, ()):
            a, b = SWAPPED_LABELS[fn]
            label = by_code[b if ann['category_id'] == a else a]  # 클래스와 알약 메타데이터를 상대 라벨의 것으로 바꾼다
        anns.append({**ann, 'category_id': label['category_id'], 'image': label['image'], 'bbox': [x, y, w, h], 'area': w * h})
    for (drawn_fn, code), (x, y, w, h) in drawn_boxes.items():
        if drawn_fn == fn:  # 빠진 알약의 메타데이터는 같은 클래스의 다른 annotation에서 가져오고, 이미지 필드만 이 이미지 값으로 바꾼다
            example = class_example[code]
            anns.append({**example, 'id': None, 'image_id': record['image']['id'], 'json_path': None, 'bbox': [x, y, w, h], 'area': w * h,
                         'image': {**example['image'], **{key: record['image'][key] for key in IMAGE_FIELDS}}})
    clean_records[fn] = {**record, 'annotations': anns}
clean_objects = [(fn, ann['category_id'], ann['bbox']) for fn, record in clean_records.items() for ann in record['annotations']]
clean_object_count = collections.Counter(code for _, code, _ in clean_objects)
assert sorted(clean_object_count) == train_classes, '정제 후 객체가 하나도 없는 클래스가 있다'

rows = [[f'`{fn}`', problem, action] for fn in sorted(problems_by_image) for problem, action in problems_by_image[fn]]
rows.append([f'**합계 {len(problems_by_image)}장**', '', '**모두 고쳐서 씀**'])
rows.append(['**정제 라벨**', f'이미지 {len(clean_records)}장, 객체 {len(clean_objects)}개, 클래스 {len(clean_object_count)}개', ''])
show_table(['이미지', '문제', '처리'], rows)

**인사이트**
- 라벨 오류가 있는 이미지는 19장(train 232장의 8.2%)이다.
    - 2-1의 8장, 2-2의 9장, 2-3의 2장이고, 서로 겹치는 이미지는 없다.
- 19장 모두 라벨을 고쳐서 쓰고, 학습에서 빼는 이미지는 없다.
    - 라벨 누락 8장: 사진에서 그린 box를 넣는다. (K-003351 4개, K-004543, K-020014, K-032310, K-033880)
    - 두 클래스의 box가 똑같은 3장: K-016548, K-020238, K-003351의 box를 box가 없던 알약으로 옮긴다.
    - K-018357의 x를 567로 고친다.
    - K-003351의 x를 185에서 385로 고친다. (2-3의 바닥을 감싼 box)
    - K-027733과 K-035206의 라벨을 서로 바꾼다.
    - K-019232의 bbox 높이를 241에서 341로 고친다.
    - K-003351의 bbox 높이를 182에서 192로 고친다.
    - 3장은 area 값만 틀려서, area를 너비×높이로 다시 계산한다. (K-012778, K-019232, K-033880, K-038162의 annotation 4개)
- 그려 넣은 box 8개는 라벨 JSON이 없으므로 annotation id와 json_path를 None으로 둔다.

**결론**
- 대회 원본 train 232장을 모두 쓰고, 그중 19장은 라벨을 고쳐서 쓴다.
- YOLO 라벨로 바꿀 때 이 목록을 그대로 적용한다. 목록은 이 셀의 규칙과 상수, 2-1과 2-2에서 그린 box(`drawn_boxes`, `moved_boxes`)로 다시 만들 수 있다.
- 정제 라벨은 이미지 232장, 객체 771개(원본 763개 + 그려 넣은 box 8개), 클래스 56개이다.
- 이후 분석 중 학습 결정에 쓰는 분석은 정제 라벨로 보고, 대회 원본 데이터 자체를 보는 분석은 원본 라벨로 본다.
    - 각 절 소개에 어느 라벨을 쓰는지 적는다.

## 3. bbox 분포
- 학습에 쓸 box의 분포를 보기 위해 2-5의 정제 라벨(이미지 232장, 객체 771개)로 본다.

### 3-1. 객체 수와 크기
- 이미지당 객체 수, bbox 너비와 높이, 입력 크기 640으로 줄였을 때의 긴 변을 본다.
    - ultralytics YOLO는 긴 변(1280)을 입력 크기(640)에 맞춰 줄인다. 그래서 box도 0.5배가 된다.
    - COCO 기준 작은 객체는 넓이 32×32픽셀 미만이다.
- 목적: 학습 입력 크기를 정한다.

In [ ]:
INPUT_SIZE = 640
scale = INPUT_SIZE / max(IMG_W, IMG_H)
clean_boxes = np.array([b for _, _, b in clean_objects], dtype=float)
objects_per_image = collections.Counter(len(r['annotations']) for r in clean_records.values())

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout='constrained')
counts = sorted(objects_per_image.items())
bars = axes[0].bar([str(k) for k, _ in counts], [v for _, v in counts])
axes[0].bar_label(bars)
axes[0].set_xlabel('이미지당 객체 수 (개)')
axes[0].set_ylabel('이미지 수 (장)')
axes[0].set_title('이미지당 객체 수')

axes[1].scatter(clean_boxes[:, 2], clean_boxes[:, 3], s=8, alpha=0.5)
lim = clean_boxes[:, 2:].max() * 1.05
for ratio in [0.5, 1, 2]:
    axes[1].plot([0, lim], [0, lim * ratio], color='gray', linestyle='--', linewidth=1)
    axes[1].text(lim / max(ratio, 1) * 0.9, lim * min(ratio, 1) * 0.9, f'높이/너비 {ratio}', fontsize=8, color='gray')
axes[1].set_xlim(0, lim)
axes[1].set_ylim(0, lim)
axes[1].set_xlabel('bbox 너비 (원본 픽셀)')
axes[1].set_ylabel('bbox 높이 (원본 픽셀)')
axes[1].set_title(f'bbox 너비와 높이 (중앙값 {np.median(clean_boxes[:, 2]):.0f}×{np.median(clean_boxes[:, 3]):.0f})')

long_side = clean_boxes[:, 2:].max(axis=1) * scale
short_side = clean_boxes[:, 2:].min(axis=1) * scale
axes[2].hist(short_side, bins=30, alpha=0.6, label='짧은 변')
axes[2].hist(long_side, bins=30, alpha=0.6, label='긴 변')
axes[2].axvline(32, color='gray', linestyle='--', linewidth=1)
axes[2].text(33, axes[2].get_ylim()[1] * 0.9, 'COCO 작은 객체 기준 32', fontsize=8, color='gray')
axes[2].set_xlabel(f'입력 {INPUT_SIZE} 기준 bbox 변 길이 (픽셀)')
axes[2].set_ylabel('객체 수')
axes[2].set_title(f'입력 {INPUT_SIZE}에서의 bbox 크기 (짧은 변 최솟값 {short_side.min():.0f}, 중앙값 {np.median(short_side):.0f})')
axes[2].legend()
fig.suptitle('bbox 개수와 크기 (2-5 정제 라벨)')
save_fig(fig, '3-1_box_size')
plt.show()

**인사이트**
- 이미지당 객체는 3개 157장, 4개 75장이다.
    - 원본에 있던 2개짜리 7장은 모두 라벨 누락 이미지이고, 빠진 box를 그려 넣어 3개가 되었다. (2-1, 2-5)
- bbox 중앙값은 230×227픽셀이고, 높이/너비는 거의 모두 0.5~2 사이이다.
- 입력 640으로 줄여도 짧은 변이 최소 62픽셀, 중앙값 107픽셀이다.
    - COCO 작은 객체 기준(32픽셀)보다 두 배 가까이 크다.

**결론**
- 학습 입력 크기는 ultralytics 기본값 640으로 시작한다.
    - 계산량은 픽셀 수에 비례한다. 입력 1280은 640보다 픽셀이 4배라, 학습 시간과 GPU 메모리도 약 4배 든다.
    - 640으로 기준선을 먼저 빠르게 만든다.
- 크기 조절(`scale`)과 모자이크 증강은 알약을 작게 만들지만, 짧은 변이 62픽셀 이상이라 기본값을 그대로 쓴다.
- 기준선을 만든 뒤 입력 1280(원본 해상도)을 같은 분할과 설정으로 학습해 640과 비교한다. 시간이 부족하면 960으로 비교한다.
    - 입력 640에서는 1픽셀이 원본 2픽셀이라, 모델이 알약 경계를 절반 해상도로 본다. IoU 0.75~0.95를 요구하는 평가 지표(2절 소개)에서 손해일 수 있다.
    - 입력 640에서는 알약 짧은 변이 중앙값 107픽셀이라 각인 글자도 원본의 절반 크기가 된다. 모양과 색이 비슷한 알약은 각인으로 구분해야 한다.
    - mAP@[0.75:0.95]와 함께 IoU 0.95의 AP(위치 정밀도)와 비슷한 알약끼리의 혼동(confusion matrix, 각인 구분)을 비교한다.
    - 오른 점수가 늘어난 학습 시간만큼의 가치가 있을 때 1280으로 바꾼다.
- 추론 입력 크기는 학습 입력 크기와 같게 둔다.
    - 640으로 학습하고 1280으로 추론하면 알약이 학습 때보다 2배 커 보여 성능이 떨어질 수 있다.
    - 예측 box는 ultralytics가 원본 976×1280 좌표로 되돌려 주므로, 입력 크기와 상관없이 제출 좌표는 원본 기준이다.

### 3-2. 위치와 겹침
- bbox 중심이 이미지의 어디에 있는지 점으로 찍는다.
- 같은 이미지 안의 box 쌍마다 IoU를 구하고, 이미지별 최댓값의 분포를 본다.
    - NMS(Non-Maximum Suppression, 많이 겹치는 예측 box 중 점수가 가장 높은 것만 남기는 후처리)는 IoU가 임계값보다 큰 box를 지운다.
    - 실제 알약끼리 IoU가 임계값보다 크게 겹치면, NMS가 맞는 box까지 지운다.
- 목적: 후처리의 NMS IoU 임계값과 최대 검출 수를 정한다.

In [ ]:
max_ious = []
for record in clean_records.values():
    boxes = [a['bbox'] for a in record['annotations']]
    pair_ious = [box_iou(boxes[i], boxes[j]) for i in range(len(boxes)) for j in range(i + 1, len(boxes))]
    max_ious.append(max(pair_ious, default=0))
max_ious = np.array(max_ious)

fig, axes = plt.subplots(1, 2, figsize=(12, 6), layout='constrained', gridspec_kw={'width_ratios': [1, 1.4]})
centers_x = clean_boxes[:, 0] + clean_boxes[:, 2] / 2
centers_y = clean_boxes[:, 1] + clean_boxes[:, 3] / 2
axes[0].scatter(centers_x, centers_y, s=8, alpha=0.5)
axes[0].set_xlim(0, IMG_W)
axes[0].set_ylim(IMG_H, 0)  # 이미지 좌표처럼 위쪽이 y=0
axes[0].set_aspect('equal')
axes[0].set_xlabel('bbox 중심 x (픽셀)')
axes[0].set_ylabel('bbox 중심 y (픽셀)')
axes[0].set_title('bbox 중심 위치 (이미지 976×1280 전체 틀)')

counts, _, bars = axes[1].hist(max_ious, bins=np.linspace(0, 1, 101))
axes[1].bar_label(bars, labels=[f'{counts[0]:.0f}'] + [''] * (len(counts) - 1), fontsize=9)
overlaps = np.sort(max_ious[max_ious > 0])
axes[1].annotate(f"IoU > 0: {len(overlaps)}장 ({', '.join(f'{v:.3f}' for v in overlaps)})", xy=(overlaps.max(), 1),
                 xytext=(0.15, 3), arrowprops=dict(arrowstyle='->', color='gray'), fontsize=9)
axes[1].set_yscale('log')  # 1장짜리 막대도 보이도록 로그 눈금
axes[1].set_ylim(0.5, counts.max() * 3)
axes[1].axvline(0.7, color='gray', linestyle='--', linewidth=1)
axes[1].text(0.7, counts.max(), ' ultralytics 기본 NMS IoU 0.7', fontsize=9, color='gray')
axes[1].set_xlabel('이미지 안 box 쌍의 IoU 최댓값 (막대 너비 0.01)')
axes[1].set_ylabel('이미지 수 (장, 로그 눈금)')
axes[1].set_title(f'이미지별 box 겹침 (최댓값 {max_ious.max():.3f}, IoU>0인 이미지 {np.mean(max_ious > 0):.1%})')
fig.suptitle('bbox 위치와 겹침 (2-5 정제 라벨)')
save_fig(fig, '3-2_box_position_overlap')
plt.show()

**인사이트**
- bbox 중심은 위쪽 3곳, 아래쪽 3곳의 여섯 자리에 모여 있다.
    - 알약을 정해진 자리에 놓고 찍은 것으로 보인다.
- 230장(99.1%)은 box끼리 전혀 겹치지 않는다.
- 겹치는 2장도 IoU 0.016, 0.021로 아주 작다.
    - 이 2장은 아래 그림에서 사진으로 확인한다.
- 원본에 있던 IoU 1.0인 3장은 2-2의 box가 똑같은 오류이고, box 하나를 옮겨 정제 라벨에서는 겹치지 않는다.

**결론**
- 실제 알약끼리는 거의 겹치지 않으므로, NMS IoU 임계값은 ultralytics 기본값 0.7로 시작한다.
    - 같은 알약에 겹쳐 나온 예측을 더 지우려고 0.5로 낮춰도 실제 알약은 지워지지 않는다. 두 값을 val에서 비교한다.
- 최대 검출 수는 기본값(300)을 그대로 쓴다.
    - 이미지당 알약은 최대 4개지만, 점수가 낮은 예측도 재현율을 올려 mAP를 높이므로 줄이지 않는다.

- box가 겹치는 2장을 사진 위에 그리고, 겹친 부분을 확대한다.
    - 왼쪽은 이미지 전체, 오른쪽은 겹친 두 box를 확대한 것이다. 노란 영역이 두 box가 겹친 곳이다.
- 실제 알약끼리 겹친 것인지, 알약 둘레의 box 여백끼리만 겹친 것인지 본다.

In [ ]:
overlap_pairs = [(fn, a, b, box_iou(a['bbox'], b['bbox']))
                 for fn, record in clean_records.items()
                 for i, a in enumerate(record['annotations']) for b in record['annotations'][i + 1:]
                 if box_iou(a['bbox'], b['bbox']) > 0]

fig, axes = plt.subplots(len(overlap_pairs), 2, figsize=(15, 7.5 * len(overlap_pairs)), layout='constrained',
                         gridspec_kw={'width_ratios': [1, 1.15]}, squeeze=False)
for row, (fn, a, b, iou) in zip(axes, overlap_pairs):
    image = Image.open(TRAIN_IMG_DIR / fn)
    for ax in row:
        ax.imshow(image)
        for ann in clean_records[fn]['annotations']:
            x, y, w, h = ann['bbox']
            color = 'red' if ann is a else 'deepskyblue' if ann is b else 'lime'
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2))
            ax.text(x, y - 6, pill_label(ann), fontsize=9, va='bottom', clip_on=True,
                    bbox=dict(facecolor=color, alpha=0.85, pad=1.5, linewidth=0))
    # 두 box가 겹친 영역을 칠하고, 두 box가 모두 들어오게 확대한다
    (xa, ya, wa, ha), (xb, yb, wb, hb) = a['bbox'], b['bbox']
    x0, y0, x1, y1 = max(xa, xb), max(ya, yb), min(xa + wa, xb + wb), min(ya + ha, yb + hb)
    row[1].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, facecolor='yellow', alpha=0.35, edgecolor='none'))
    row[1].set_xlim(min(xa, xb) - 30, max(xa + wa, xb + wb) + 30)
    row[1].set_ylim(max(ya + ha, yb + hb) + 30, min(ya, yb) - 50)
    row[0].set_title(f'{fn}\n빨강·파랑 = 겹친 두 box, 연두 = 나머지 box', fontsize=10)
    row[0].set_xticks([])
    row[0].set_yticks([])
    row[1].set_title(f'겹친 부분 확대 (노란 영역 = 두 box가 겹친 곳 {x1 - x0}×{y1 - y0}픽셀, IoU {iou:.3f})\n축 = 원본 픽셀 좌표', fontsize=10)
fig.suptitle(f'box가 겹치는 이미지 {len(overlap_pairs)}장 (2-5 정제 라벨)', fontsize=14)
save_fig(fig, '3-2b_overlap_pairs')
plt.show()

**인사이트**
- 2장은 같은 조합(`K-003351-033880-035206`)을 75°와 90°에서 찍은 사진이다.
- 두 장 모두 JIC 정제(K-033880)와 337 정제(K-035206)가 맞닿아 놓여 있다.
- 알약끼리는 겹치지 않는다.
    - JIC의 오른쪽 끝과 337의 왼쪽 끝이 거의 닿아서, 알약 둘레의 box 여백끼리 좁은 세로 띠로 겹친다.
    - 겹친 띠는 75° 사진이 22×215픽셀, 90° 사진이 17×197픽셀이다.
- 두 box 모두 자기 알약을 감싸고 있어 라벨 오류가 아니다.

**결론**
- 알약이 맞닿아 놓여도 box IoU는 0.02 수준이라, NMS 임계값 0.7(낮춰도 0.5)에서 맞는 box가 지워지지 않는다.
- 3-2 결론대로 NMS IoU 임계값은 0.7로 시작한다.

## 4. 클래스 분포와 각도 묶음
- 목적: 분할 단위(이미지 또는 조합), 클래스 불균형 대응, 회전과 반전 증강 사용 여부를 정한다.
- 4-1은 학습에 쓸 클래스 분포라 2-5의 정제 라벨로 보고, 4-2와 4-3은 대회가 준 사진 구성을 보므로 원본 232장으로 본다.

### 4-1. 클래스별 객체 수와 등장 조합 수
- 클래스마다 객체 수와, 그 클래스가 나오는 조합 수를 같은 순서로 그린다.
    - 조합(한 번에 함께 놓고 찍은 알약 묶음(3~4개)) 단위로 나누면 같은 조합의 사진은 train과 val 중 한쪽에만 들어간다.
    - 그래서 1조합에만 나오는 클래스는 train과 val 양쪽에 둘 수 없는 문제가 생긴다. 즉, 한 split에만 들어가서 훈련이 제대로 되지 않는 경우가 생긴다. 

In [ ]:
combos_by_class = collections.defaultdict(set)
for fn, code, _ in clean_objects:
    combos_by_class[code].add(combo_name(fn))
order = sorted(train_classes, key=lambda c: (-clean_object_count[c], c))
labels = [f'K-{c:06d}' for c in order]
single_combo = [c for c in train_classes if len(combos_by_class[c]) == 1]

fig, axes = plt.subplots(2, 1, figsize=(18, 8), sharex=True, layout='constrained')
bars = axes[0].bar(labels, [clean_object_count[c] for c in order])
axes[0].bar_label(bars, fontsize=7)
axes[0].set_ylabel('객체 수 (개)')
axes[0].set_title(f'클래스별 객체 수 (전체 {len(clean_objects)}개)')
colors = ['tab:red' if len(combos_by_class[c]) == 1 else 'tab:blue' for c in order]
bars = axes[1].bar(labels, [len(combos_by_class[c]) for c in order], color=colors)
axes[1].bar_label(bars, fontsize=7)
axes[1].set_ylabel('등장 조합 수 (개)')
axes[1].set_title(f'클래스별 등장 조합 수 (빨강 = 1조합에만 나오는 클래스 {len(single_combo)}개, 전체 조합 {len({combo_name(fn) for fn in clean_records})}개)')
axes[1].tick_params(axis='x', rotation=90, labelsize=8)
fig.suptitle('클래스별 객체 수와 등장 조합 수 (2-5 정제 라벨, 객체가 많은 순서)')
save_fig(fig, '4-1_class_counts')
plt.show()

**인사이트**
- K-003351이 객체 157개(전체 771개의 20%)로 가장 많고, 114조합 중 89조합에 나온다.
- 그다음은 K-003483 45개, K-035206 37개이고, 나머지 53개 클래스는 23개 이하이다.
- 17개 클래스는 1조합에만 나온다. (빨간 막대)
    - 17개 모두 객체가 3개이다.
    - 객체가 모두 같은 조합의 각도별 사진에 있어서, 사실상 장면 하나뿐이다.

**결론**
- 대회 원본만으로는 조합 단위로 나눌 때 17개 클래스를 val에 넣을 수 없다. 이 클래스들을 보충 1순위로 둔다.
- K-003351은 89조합에 걸쳐 있어 이미지를 빼서 줄일 수 없으므로, 줄이지 않는다.
- 클래스 가중치나 샘플링 같은 불균형 대응은 보충 후의 분포를 보고 정한다.

### 4-2. 조합당 이미지 수
- 조합 = 한 번에 함께 놓고 찍은 알약 묶음(3~4개).
    - 조합 이름은 이미지 파일명에서 첫 _ 앞부분이다(combo_name).
    - 예: K-001900-016548-019607-029451_0_2_0_2_70_000_200.png의 조합은 K-001900-016548-019607-029451이다.
    - 이 사진에는 K-001900, K-016548, K-019607, K-029451 알약 4개가 함께 놓여 있다는 뜻이다.
- 조합 하나에 사진이 몇 장 있는지 센다.
    - AI Hub 원본은 조합마다 3장(카메라 위도 70°, 75°, 90°)의 이미지가 있다고 소개한다. (AI Hub 데이터셋 소개 페이지)
- 사진별 촬영 각도(파일명의 여섯 번째 값, camera_la와 같음)도 센다.

In [ ]:
images_per_combo = collections.Counter(collections.Counter(combo_name(fn) for fn in train_files).values())
angle_counts = collections.Counter(fn.split('_')[5] for fn in train_files)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
items = sorted(images_per_combo.items())
bars = axes[0].bar([f'{k}장' for k, _ in items], [v for _, v in items])
axes[0].bar_label(bars)
axes[0].set_xlabel('조합당 이미지 수')
axes[0].set_ylabel('조합 수 (개)')
axes[0].set_title(f'조합당 이미지 수 (조합 {len(train_combos)}개, 3장 기준이면 {len(train_combos) * 3}장 중 {len(train_files)}장)')
items = sorted(angle_counts.items())
bars = axes[1].bar([f'{k}°' for k, _ in items], [v for _, v in items], color='tab:orange')
axes[1].bar_label(bars)
axes[1].set_xlabel('카메라 위도')
axes[1].set_ylabel('이미지 수 (장)')
axes[1].set_title('촬영 각도별 이미지 수')
fig.suptitle('조합과 촬영 각도')
save_fig(fig, '4-2_images_per_combo')
plt.show()

**인사이트**
- 조합 114개 중 사진이 1장인 조합 36개, 2장 38개, 3장 40개이다.
    - AI Hub 원본은 조합마다 3장이므로, 342장 중 232장만 train에 있고 110장이 빠졌다.
- 촬영 각도는 70° 72장, 75° 75장, 90° 85장으로 고르다.

**결론**
- 각도별 비율이 고르므로, 각도를 맞춰 샘플링하지 않는다.

### 4-3. 같은 조합 사진 비교
- 사진이 3장인 조합 하나를 골라 각도별 사진을 box와 함께 나란히 본다.
    - 같은 장면을 다시 찍은 것이면, 이미지 단위로 나눌 때 val에 train과 거의 같은 사진이 들어간다.
        - 이런 사진은 파일 내용이 달라서 2-4의 해시 비교로는 잡히지 않는다.
    - 알약과 각인이 어떻게 놓였는지도 본다. 좌우 반전 증강은 각인을 거울상으로 뒤집는다.

In [ ]:
EXAMPLE_COMBO = 'K-001900-016548-019607-029451'
example_files = sorted(fn for fn in train_files if combo_name(fn) == EXAMPLE_COMBO)
fig, axes = plt.subplots(1, len(example_files), figsize=(5.2 * len(example_files), 7), layout='constrained')
for ax, fn in zip(axes, example_files):
    anns = train_records[fn]['annotations']
    ax.imshow(draw_boxes(TRAIN_IMG_DIR / fn, [a['bbox'] for a in anns], [a['category_id'] for a in anns], width=6))
    ax.set_title(f'카메라 위도 {fn.split("_")[5]}°', fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(f'같은 조합 {EXAMPLE_COMBO}의 각도별 사진 (숫자 = K-코드)')
save_fig(fig, '4-3_same_combo')
plt.show()

**인사이트**
- 세 장 모두 같은 알약 4개를 같은 배치로 찍은 사진이다.
- 75°와 90° 사진은 알약 위치와 방향이 거의 같다.
- 70° 사진은 배치 전체가 180° 돌아가 있다. 각인(BSP, GB, LAYLA)도 함께 180° 돌아가 있고, 거울상이 된 각인은 없다.

**결론**
- 분할은 조합 단위로 한다.
    - 이미지 단위로 나누면 75°와 90°처럼 거의 같은 사진이 train과 val에 나뉘어 들어가, val 점수가 실제보다 높게 나올 수 있다.
- 좌우 반전(`fliplr`, ultralytics 기본값 0.5)과 상하 반전(`flipud`)은 쓰지 않는다.
    - 반전은 각인을 거울상으로 만드는데, 실제 사진에는 거울상 각인이 없다.
    - 모양과 색이 비슷하고 각인만 다른 알약(예: 흰색 원형 정제)은 각인이 중요한 단서이다.
- 임의 각도 회전(`degrees`)은 쓰지 않는다.
    - 돌린 box를 다시 축에 맞춰 감싸므로 box가 헐거워지고, IoU 0.75~0.95를 요구하는 평가 지표(2절 소개)에서 손해이다.
- 180° 회전은 실제로 있는 변화이지만 ultralytics에 해당하는 옵션이 없다. 학습 이미지를 180° 돌린 복사본을 미리 만들어 넣는 방법을 추가 실험으로 둔다.

## 5. test 알약 직접 확인
- test는 라벨이 없어서, train에 없는 알약이 test에 있는지는 사진을 직접 봐야 안다.
- train 56개 클래스의 갤러리(5-1)를 옆에 두고, test 842장을 한 장씩 보며 갤러리에 없는 알약을 기록표에 적는다. (5-2, 5-3)
    - 1차 확인은 Claude Code가 하고, 2차 확인은 사람과 Claude Code가 나눠서 한다. (5-2)
- 목적: test에만 있는 클래스를 확정해 보충 범위와 클래스 체계를 정한다.

### 5-1. 찾아보기용 train 갤러리
- 56개 클래스를 색(`color_class1`)과 모양(`drug_shape`)으로 묶는다. test 알약을 보고 같은 색, 같은 모양 칸만 찾아보면 된다.
- 칸마다 crop 2개와 K-코드, 제품명, 앞뒤 각인(`print_front`, `print_back`)을 붙인다.
    - crop은 서로 다른 조합에서 먼저 고르고, 조합이 하나뿐인 클래스는 같은 조합의 다른 사진에서 고른다.
    - crop은 box 중심에서 긴 변보다 10% 큰 정사각형으로 잘라, 알약 모양이 늘어나지 않게 한다.
    - crop은 2-5의 정제 라벨에서 고른다. 라벨을 고친 19장은 고친 라벨로 쓴다.

In [ ]:
GALLERY_CROPS = 2  # 클래스마다 보여 줄 crop 수
CROP_SIZE = 200  # crop 한 변의 길이 (픽셀)

candidates = collections.defaultdict(list)  # K-코드 → [(파일명, bbox, 메타데이터)]
for fn, record in clean_records.items():
    for ann in record['annotations']:
        candidates[ann['category_id']].append((fn, ann['bbox'], ann['image']))

gallery_picks = {}
for code, items in candidates.items():
    first_of_combo = {}
    for item in items:
        first_of_combo.setdefault(combo_name(item[0]), item)
    chosen = list(first_of_combo.values())[:GALLERY_CROPS]
    chosen += [item for item in items if item not in chosen][:GALLERY_CROPS - len(chosen)]
    gallery_picks[code] = chosen

# K-코드 → 갤러리에 적을 메타데이터 (빈 값은 '-')
FIELDS = ['color_class1', 'drug_shape', 'dl_name', 'print_front', 'print_back']
pill_info = {code: {k: items[0][2][k] or '-' for k in FIELDS} for code, items in gallery_picks.items()}
color_size = collections.Counter(pill_info[c]['color_class1'] for c in gallery_picks)
gallery_order = sorted(gallery_picks, key=lambda c: (-color_size[pill_info[c]['color_class1']], pill_info[c]['color_class1'],
                                                     pill_info[c]['drug_shape'], c))

n_cols = 8
n_rows = int(np.ceil(len(gallery_order) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(24, n_rows * 3.5), layout='constrained')
for ax, code in zip(axes.flat, gallery_order):
    crops = [crop_square(Image.open(TRAIN_IMG_DIR / fn).convert('RGB'), bbox, size=CROP_SIZE) for fn, bbox, _ in gallery_picks[code]]
    ax.imshow(np.concatenate([np.asarray(c) for c in crops], axis=1))
    info = pill_info[code]
    ax.set_title(f"{info['color_class1']} {info['drug_shape']} | K-{code:06d}\n{info['dl_name'][:16]}\n"
                 f"각인 {info['print_front']} / {info['print_back']}", fontsize=9)
for ax in axes.flat:
    ax.axis('off')
fig.suptitle(f'찾아보기용 train 갤러리 {len(gallery_order)}개 클래스 (색, 모양 순서로 묶음, 칸마다 crop {GALLERY_CROPS}개)\n'
             '칸 제목 = 색 모양 | K-코드, 제품명, 각인 앞 / 뒤', fontsize=14)
save_fig(fig, '5-1_lookup_gallery')
plt.show()

**인사이트**
- 56개 클래스는 색 9가지로 나뉜다. 하양 19개, 분홍 10개, 노랑 10개, 갈색 6개, 주황 5개 순이다.
- 흰색 원형 7개는 생김새가 비슷해서 각인으로만 구분된다. (OG33, ALT, Airtal, FZ12, GIL 1, ATV, LPL)
- K-016548(GB 알약)은 메타데이터가 비어 있어 색, 모양, 각인이 '-'로 표시된다.

**결론**
- test 알약을 볼 때 같은 색 칸을 먼저 찾고, 각인으로 클래스를 확인한다.
- 각인이 갤러리에 없으면 처음 보는 알약으로 기록한다.

### 5-2. test 확인 시트
- test 842장을 절반 크기로 줄여 한 장에 12장(4×3)씩 붙이고, `runs/eda_raewoo/test_sheets/`에 저장한다. 사진 위의 글자는 test 파일명이다.
- 시트를 한 장씩 보며 5-1 갤러리에 없는 알약을 찾고, 기록표 `notebooks/eda/00_raewoo_test_review.csv`에 알약 하나당 한 줄씩 적는다.
    - 기록표는 직접 적은 표라 다시 만들 수 없어서, git에서 제외되는 `runs/` 대신 노트북 옆에 두고 git으로 공유한다.
    - 각인을 시트에서 읽기 어려우면 원본 사진(`data/raw/test_images/<번호>.png`)을 열어 읽는다.
    - 기록표의 열
        - `test_file`: test 파일명
        - `imprint`: 보이는 각인. 분할선은 `|`로 적고, 읽을 수 없으면 `?`로 적는다.
        - `color`, `shape`: 5-1 갤러리와 같은 말로 적는다. (예: 하양, 원형)
        - `position`: 사진 안 위치 (예: 왼쪽 위)
        - `note`: 메모
        - `checked`: 2차 확인에서 맞으면 O, 틀렸으면 X
    - 기록표가 없으면 이 셀이 열 이름만 있는 빈 기록표를 만든다.
- 아래 그림은 첫 번째 시트이다.

In [ ]:
SHEET_DIR = FIG_DIR.parent / 'test_sheets'
REVIEW_CSV = PROJECT_ROOT / 'notebooks' / 'eda' / '00_raewoo_test_review.csv'  # 직접 적은 기록표라 git으로 공유한다
REVIEW_COLUMNS = ['test_file', 'imprint', 'color', 'shape', 'position', 'note', 'checked']
SHEET_COLS, SHEET_ROWS = 4, 3
THUMB_W, THUMB_H = IMG_W // 2, IMG_H // 2
LABEL_H = 44  # 사진 위 파일명 칸의 높이 (픽셀)

SHEET_DIR.mkdir(exist_ok=True)
per_sheet = SHEET_COLS * SHEET_ROWS
font = ImageFont.load_default(size=34)
sheet_paths = []
for start in range(0, len(test_files), per_sheet):
    sheet = Image.new('RGB', (SHEET_COLS * THUMB_W, SHEET_ROWS * (THUMB_H + LABEL_H)), 'white')
    draw = ImageDraw.Draw(sheet)
    for k, fn in enumerate(test_files[start:start + per_sheet]):
        x, y = (k % SHEET_COLS) * THUMB_W, (k // SHEET_COLS) * (THUMB_H + LABEL_H)
        sheet.paste(Image.open(TEST_IMG_DIR / fn).convert('RGB').resize((THUMB_W, THUMB_H)), (x, y + LABEL_H))
        draw.text((x + 8, y + 4), fn, fill='black', font=font)
    sheet_paths.append(SHEET_DIR / f'sheet_{start // per_sheet + 1:02d}.png')
    sheet.save(sheet_paths[-1])

if not REVIEW_CSV.exists():
    with open(REVIEW_CSV, 'w', newline='', encoding='utf-8') as f:
        csv.writer(f).writerow(REVIEW_COLUMNS)

fig, ax = plt.subplots(figsize=(14, 15), layout='constrained')
ax.imshow(Image.open(sheet_paths[0]))
ax.axis('off')
ax.set_title(f'test 확인 시트 1/{len(sheet_paths)} (한 장에 {per_sheet}장, 전체는 {SHEET_DIR.relative_to(PROJECT_ROOT)})')
save_fig(fig, '5-2_test_sheet_example')
plt.show()

**인사이트**
- test 842장이 시트 71장으로 저장되었다. 마지막 시트에는 2장이 있다.
- 절반 크기로 줄여도 대부분의 각인을 읽을 수 있다.
    - 작은 알약(Q25)과 빨간 장방형 알약(V100, MG, B-F, Y|H)은 원본 사진을 잘라 확대해야 구분된다.
- 1차 확인(Claude Code)으로 처음 보는 알약 442개를 기록표에 적었다.
- 2차 확인에서 틀린 줄은 없었고, 1차에서 놓친 알약 11개를 더 찾았다. 기록표는 453줄이 되었다.
    - 사람이 앞 64줄을 확인했다. Claude Code는 나머지 378줄을 각인별로 모아, 적힌 위치를 잘라 다시 봤다.
    - 놓친 알약은 C77 7개(605, 610, 611, 612, 692, 1138, 1140번)와 Hanmi 4개(719, 1256, 1279, 1280번)이다. 모두 이미 찾은 각인이라 새 각인은 없다.
- 놓친 알약은 같은 조합의 사진끼리 비교해 찾았다.
    - test 파일 번호와 train 이미지 id는 같은 번호 체계이고, 1~3번, 4~6번처럼 번호 3개가 한 조합이다.
        - train 232장에서 같은 묶음에 든 사진은 조합 이름이 모두 같았다.
    - test 842장 중 110장은 train 조합의 다른 각도 사진이라 알약이 모두 train 클래스이다. 나머지 732장은 test에만 있는 조합 384개이다.
        - 이 110장은 4-2에서 train에 빠져 있던 각도 사진 110장이다.
    - 같은 조합인데 사진마다 기록이 다른 조합은 하나(719, 720번)였고, 719번의 Hanmi를 놓친 것이었다.
    - test에만 있는 조합 384개는 조합마다 사진 1장을 다시 보고, 놓친 알약이 나온 조합은 나머지 사진도 봤다.

**결론**
- 기록표는 2차 확인까지 끝난 453줄로 확정한다.
- 같은 조합의 사진은 알약이 같으므로, 기록표를 고칠 때는 번호 3개씩 묶은 같은 조합의 사진도 함께 고친다.
- test에 train 조합의 다른 각도 사진 110장이 있어도, 분할은 4-3대로 train 안에서 조합 단위로 한다.
    - test의 나머지 732장(87%)은 train에 없는 조합이라, 조합 단위로 나눈 val이 이 사진들과 같은 조건이다.

### 5-3. 처음 보는 알약 목록
- 기록표를 읽어, 각인과 색과 모양이 같은 알약끼리 묶고 몇 장의 test 사진에 나오는지 센다.
- `2차 확인` 열은 2차 확인(5-2)에서 O를 받은 줄 수이다.

In [ ]:
with open(REVIEW_CSV, encoding='utf-8') as f:
    review_rows = list(csv.DictReader(f))
if not review_rows:
    raise ValueError(f'기록표가 비어 있다: {REVIEW_CSV}\n'
                     '5-2의 시트를 보고 처음 보는 알약을 한 줄씩 적은 뒤 이 셀을 다시 실행한다.')

review_groups = collections.defaultdict(list)  # (각인, 색, 모양) → 기록표 줄
for row in review_rows:
    review_groups[(row['imprint'], row['color'], row['shape'])].append(row)
rows = []
for (imprint, color, shape), items in sorted(review_groups.items(), key=lambda kv: (-len(kv[1]), kv[0])):
    numbers = sorted({int(row['test_file'].split('.')[0]) for row in items})
    shown = ', '.join(map(str, numbers[:8])) + (' …' if len(numbers) > 8 else '')
    rows.append([imprint, color, shape, len(numbers), shown, f"{sum(row['checked'] == 'O' for row in items)}/{len(items)}"])
images_with_new = {row['test_file'] for row in review_rows}
rows.append([f'**합계 {len(review_groups)}종**', '', '',
             f'**{len(images_with_new)}장 ({len(images_with_new) / len(test_files):.1%})**', '', ''])
show_table(['각인', '색', '모양', 'test 사진 수', 'test 파일 번호', '2차 확인 (O)'], rows)

**인사이트**
- train 갤러리에 없는 알약은 각인 기준 19종이고, test 842장 중 335장(39.8%)에 1개 이상 나온다.
    - 가장 많은 것은 C77(73장), AHG(52장), B-F(31장)이다.
    - 213과 30은 같은 알약의 앞면과 뒷면으로 본다. 둘 다 흰 바탕에 주황 점이 박힌 같은 크기의 원형 정제이다. 그래서 알약은 18종이다.
    - 19종의 사진은 아래 그림에서 확인한다.
- train에 각인이 비슷한 알약이 있어도 다른 알약이다.
    - 노란 타원형 SK는 train의 초록 장방형 SK(K-003483, 뒷면 G40)와 다르다.
    - TYLENOL과 TYLENOL ER은 서로 다른 알약이고, 둘 다 train에 없다.
- 2차 확인까지 끝난 결과이다. 453줄이 모두 O이다. (사람 64줄, Claude Code 389줄)

**결론**
- test의 약 40% 사진에 train에 없는 알약이 있으므로, 이 알약들을 보충하지 않으면 그 객체는 모두 놓친다.
- 각인 19종의 K-코드는 허용 AI Hub 라벨의 각인 필드에서 찾는다.

- 기록표의 각인 19종을 test 사진으로 보여 준다.
    - 각인마다 서로 다른 조합의 사진 2장을 파일 번호 순서로 고른다. (번호 3개가 한 조합, 5-2)
- 기록표에는 box 없이 위치(사분면)만 있으므로, 그 위치 쪽 모서리에서 620×780픽셀을 자른다.
    - 사분면(488×640)보다 크게 잘라, 사분면 경계에 걸친 알약도 잘리지 않게 한다.
    - 옆 알약 일부가 함께 보일 수 있다. 칸 제목의 색과 모양으로 해당 알약을 찾는다.
- 기록표의 각인이 실제 알약과 맞는지, 생김새가 비슷해 각인으로만 구분되는 알약이 있는지 본다.

In [ ]:
PHOTO_EXAMPLES = 2  # 각인마다 보여 줄 test 사진 수
POS_CROP_W, POS_CROP_H = 620, 780  # 기록표 위치 쪽 모서리에서 자를 크기 (사분면 488×640보다 조금 크게)


def crop_position(image, position):
    """기록표의 위치(예: 왼쪽 위) 쪽 모서리에서 POS_CROP_W×POS_CROP_H 크기로 자른다."""
    x0 = 0 if '왼쪽' in position else IMG_W - POS_CROP_W
    y0 = 0 if '위' in position else IMG_H - POS_CROP_H
    return image.crop((x0, y0, x0 + POS_CROP_W, y0 + POS_CROP_H))


photo_order = sorted(review_groups, key=lambda key: (-len(review_groups[key]), key))  # 5-3 표와 같은 순서
n_cols = 4
n_rows = int(np.ceil(len(photo_order) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(24, n_rows * 4.6), layout='constrained')
for ax, key in zip(axes.flat, photo_order):
    first_of_combo = {}  # 조합 → 그 조합에서 파일 번호가 가장 작은 줄 (번호 3개가 한 조합, 5-2)
    for row in sorted(review_groups[key], key=lambda r: int(r['test_file'].split('.')[0])):
        first_of_combo.setdefault((int(row['test_file'].split('.')[0]) - 1) // 3, row)
    picks = list(first_of_combo.values())[:PHOTO_EXAMPLES]
    crops = [np.asarray(crop_position(Image.open(TEST_IMG_DIR / row['test_file']).convert('RGB'), row['position']))
             for row in picks]
    ax.imshow(np.concatenate(crops, axis=1))
    for k in range(1, len(crops)):
        ax.axvline(k * POS_CROP_W, color='white', linewidth=6)  # 사진 경계
    imprint, color, shape = key
    n_photos = len({row['test_file'] for row in review_groups[key]})
    where = ', '.join(f"{row['test_file'].split('.')[0]}번 {row['position']}" for row in picks)
    ax.set_title(f'{imprint} ({color} {shape}, test {n_photos}장)\n{where}', fontsize=11)
for ax in axes.flat:
    ax.axis('off')
fig.suptitle(f'처음 보는 알약 {len(photo_order)}종의 test 사진 (각인마다 서로 다른 조합 {PHOTO_EXAMPLES}장, 기록표 위치 쪽 모서리를 자름)\n'
             '칸 제목 = 각인 (색 모양, 나오는 test 사진 수), 아래 줄 = 고른 test 파일 번호와 기록표 위치', fontsize=14)
save_fig(fig, '5-3b_new_pill_photos')
plt.show()

**인사이트**
- 19종 모두 기록표에 적은 위치에 해당 알약이 있고, 사진의 각인이 기록표와 같다.
    - 그림에서 작게 보이는 B-F, TL, Q25, AEI도 원본 사진에서는 각인이 읽힌다.
- 213과 30은 사진에서도 흰 바탕에 주황 점이 박힌 같은 크기의 원형 정제이다.
- 색과 모양이 비슷해 각인으로만 구분되는 알약이 있다.
    - 빨간 장방형 B-F, V100, MG는 색과 크기가 비슷하다.
    - TYLENOL과 TYLENOL ER은 모양이 같고, 각인의 ER 한 줄만 다르다.
- Q25는 19종 중 가장 작은 알약이다.

**결론**
- 기록표의 19종은 사진으로도 맞으므로, 5-3의 목록을 그대로 쓴다.
- 각인으로만 구분되는 알약(빨간 장방형 B-F, V100, MG와 TYLENOL, TYLENOL ER)은 학습 후 val에서 서로 헷갈리는지 따로 확인한다.
    - 3-1의 입력 크기 비교(640과 1280)에서 이 알약들의 혼동을 함께 본다. 입력을 줄이면 각인 글자도 함께 작아진다.

## 6. AI Hub 조합 데이터로 보충
- 대회 원본 데이터의 출처는 AI Hub 576 "약품식별 인공지능 개발을 위한 경구약제 이미지 데이터"이다. (DATA.md)
- 대회 규칙은 이 중 `TL_2_조합.zip`, `TS_2_조합.zip` 두 파일만 금지한다. (RULE.md)
- 목적: 보충할 데이터와 변환 방법을 정한다.

### 6-1. 파일 구성과 금지 범위
- `aihubshell -mode l -datasetkey 576`으로 받은 파일 목록을 정리한다. (2026-10-24 조회)
    - TL/VL은 라벨링 데이터(JSON), TS/VS는 원천 데이터(이미지)이다. T는 Training, V는 Validation이다.
    - 용량은 aihubshell이 반올림해 보여 준 값이다.

In [ ]:
AIHUB_FILES = [
    # (구분, 파일, 개수, 파일 하나 용량, filekey, 사용)
    ('Training 조합 라벨', 'TL_1, TL_3~8_조합.zip', 7, '7~8MB', '66065, 66067~66072', '사용 가능'),
    ('Training 조합 라벨', 'TL_2_조합.zip', 1, '7MB', '66066', '**금지**'),
    ('Training 조합 이미지', 'TS_1, TS_3~8_조합.zip', 7, '3GB', '66154, 66156~66161', '사용 가능'),
    ('Training 조합 이미지', 'TS_2_조합.zip', 1, '3GB', '66155', '**금지**'),
    ('Validation 조합 라벨', 'VL_1_조합.zip', 1, '7MB', '66243', '사용 가능'),
    ('Validation 조합 이미지', 'VS_1_조합.zip', 1, '3GB', '66254', '사용 가능'),
    ('Training 단일 라벨', 'TL_1~81_단일.zip', 81, '11~73MB (합계 2.6GB)', '66073~66153', '사용 가능'),
    ('Training 단일 이미지', 'TS_1~81_단일.zip', 81, '13~95GB (합계 3,473GB)', '66162~66242', '사용 가능 (로컬 디스크 부족)'),
    ('Validation 단일 라벨', 'VL_1~10_단일.zip', 10, '15~24MB', '66244~66253', '사용 가능'),
    ('Validation 단일 이미지', 'VS_1~10_단일.zip', 10, '19~31GB (합계 228GB)', '66255~66264', '사용 가능 (로컬 디스크 부족)'),
]
show_table(['구분', '파일', '개수', '용량', 'filekey', '사용'], [list(row) for row in AIHUB_FILES])

**인사이트**
- 조합 데이터는 라벨 zip 9개와 이미지 zip 9개로 나뉘어 있고, 그중 2번 한 쌍만 금지이다.
- 사용 가능한 조합 데이터는 라벨 zip 8개(각 7~8MB)와 이미지 zip 8개(각 3GB)이다.
- 단일 데이터는 이미지 zip이 각 13~95GB, 합계 약 3.7TB이다.

**결론**
- 보충은 허용된 조합 데이터 8쌍으로 한다.
    - test와 같은 조합 사진이고, zip 하나가 3GB라 로컬 디스크(남은 공간 약 23GiB)로 받을 수 있다.
- 단일 이미지 zip은 로컬에서 받지 않는다. 조합 데이터에 없는 클래스가 있으면, 별도 환경을 정해 단일 데이터로 다룬다.
- 금지 파일은 공용 모듈의 `check_aihub_file`이 파일 이름으로 막는다.

### 6-2. 라벨 구조 비교
- 받은 허용 조합 라벨 zip 8개(TL_1, TL_3~8, VL_1)를 풀지 않고 읽어, Kaggle train 라벨과 같은 항목을 비교한다.
    - dl_idx는 알약 식별자 필드이고, drug_N은 K-코드(`K-016548`)이다.
- JSON 키 구성은 최상위, images, annotations, categories 각각의 키 이름을 순서대로 모아 Kaggle과 같은지 센다.

In [ ]:
aihub_zips = sorted(AIHUB_DIR.rglob('*.zip'))
for path in aihub_zips:
    assert unicodedata.normalize('NFC', path.name) not in ('TL_2_조합.zip', 'TS_2_조합.zip'), f'금지 파일이 있다: {path}'
label_zips = [path for path in aihub_zips  # 라벨 zip은 train_annotations/에 있다 (script/download_aihub.py의 폴더 규칙)
              if path.parent.name == 'train_annotations' and unicodedata.normalize('NFC', path.name).endswith('_조합.zip')]
assert len(label_zips) == 8, (f'허용 조합 라벨 zip 8개가 {AIHUB_DIR / "train_annotations"}에 있어야 한다. 지금 {len(label_zips)}개\n'
                              '받는 명령: uv run python script/download_aihub.py 66065 66067 66068 66069 66070 66071 66072 66243')


def key_structure(data):
    """JSON의 최상위, images, annotations, categories에 있는 키 이름을 순서대로 모은다."""
    return tuple(tuple(part) for part in (data, data['images'][0], data['annotations'][0], data['categories'][0]))


kaggle_structures = collections.Counter(key_structure(json.loads(p.read_text(encoding='utf-8')))
                                        for p in TRAIN_ANN_DIR.rglob('*.json'))
kaggle_structure = kaggle_structures.most_common(1)[0][0]
raw = collections.Counter()
aihub_ids = {'image': collections.Counter(), 'category': collections.Counter(), 'annotation': collections.Counter(),
             'category_name': collections.Counter()}
for path in label_zips:
    with zipfile.ZipFile(path) as zf:
        for name in zf.namelist():
            name_nfc = unicodedata.normalize('NFC', name)
            if not name_nfc.endswith('.json'):
                continue
            base = os.path.basename(name_nfc)
            if base.startswith('._'):
                raw['macOS ._ 파일'] += 1
                continue
            raw['JSON 파일'] += 1
            if ' copy' in base:
                raw['copy 중복 파일'] += 1
            if name_nfc.split('/')[-2].endswith('-오류'):
                raw['-오류 폴더 JSON'] += 1
            try:
                data = json.loads(zf.read(name))
            except json.JSONDecodeError:
                raw['읽을 수 없는 JSON'] += 1
                continue
            image = data['images'][0]
            raw['키 구성 = Kaggle'] += key_structure(data) == kaggle_structure
            k_code = int(image['drug_N'][2:])
            raw['dl_idx = K-코드'] += int(image['dl_idx']) == k_code
            raw['dl_idx = K-코드 - 1'] += int(image['dl_idx']) == k_code - 1
            raw['폴더 이름 = drug_N'] += name_nfc.split('/')[-2] == image['drug_N']
            aihub_ids['image'][image['id']] += 1
            aihub_ids['category_name'][data['categories'][0]['name']] += 1
            for ann in data['annotations']:
                aihub_ids['category'][ann.get('category_id')] += 1
                aihub_ids['annotation'][ann.get('id')] += 1

kaggle_anns = [ann for r in train_records.values() for ann in r['annotations']]
n_valid = raw['JSON 파일'] - raw['읽을 수 없는 JSON']


def top_values(counter):
    """가장 많은 값 두 개를 '값 (개수)' 문자열로 만든다."""
    return ', '.join(f'{v} ({n})' for v, n in counter.most_common(2))


rows = [
    ['폴더 구조', '`<조합>_json/K-<코드>/<이미지명>.json`', '같음 (폴더 이름 = drug_N ' + f"{raw['폴더 이름 = drug_N']}/{n_valid})"],
    ['JSON 키 구성 (최상위, images, annotations, categories)',
     f"한 가지 {kaggle_structures[kaggle_structure]}/{json_count} (키 {', '.join(str(len(p)) for p in kaggle_structure)}개)",
     f"Kaggle과 같음 {raw['키 구성 = Kaggle']}/{n_valid}"],
    ['JSON 파일 수', len(kaggle_anns), raw['JSON 파일']],
    ['image id', f'이미지마다 다름 ({len({r["image"]["id"] for r in train_records.values()})}종)', top_values(aihub_ids['image'])],
    ['category_id', f'K-코드 숫자 ({len(train_classes)}종)', top_values(aihub_ids['category'])],
    ['annotation id', f'annotation마다 다름 ({len({a["id"] for a in kaggle_anns})}종)', top_values(aihub_ids['annotation'])],
    ['categories의 name', '제품명 (dl_name)', top_values(aihub_ids['category_name'])],
    ['dl_idx = K-코드', f"{sum(int(a['image']['dl_idx']) == a['category_id'] for a in kaggle_anns)}/{len(kaggle_anns)}",
     f"{raw['dl_idx = K-코드']}/{n_valid}"],
    ['dl_idx = K-코드 - 1', '0', f"{raw['dl_idx = K-코드 - 1']}/{n_valid}"],
    ['-오류 폴더 JSON', '0', raw['-오류 폴더 JSON']],
    ['읽을 수 없는 JSON', '0', raw['읽을 수 없는 JSON']],
    ['copy 중복 파일', '0', raw['copy 중복 파일']],
    ['macOS ._ 파일', '0', raw['macOS ._ 파일']],
]
show_table(['항목', 'Kaggle train', 'AI Hub 조합 라벨 (허용 8개)'], rows)

**인사이트**
- 폴더 구조는 Kaggle과 같다. 폴더 이름이 drug_N과 다른 JSON 8개는 모두 "-오류" 폴더에 있다.
- JSON 키 구성도 Kaggle과 같다. 45,933개 중 45,932개가 최상위 4개, images 50개, annotations 8개, categories 3개 키를 같은 이름, 같은 순서로 가진다.
    - 나머지 1개는 annotations에 `iscrowd` 키가 없다.
- AI Hub 원본은 image id, category_id, annotation id가 모두 1이고, categories의 name도 모두 "Drug"이다.
    - DATA.md의 "원본은 id가 모두 1"이라는 설명과 같다.
- dl_idx가 K-코드와 같은 JSON은 하나도 없다.
    - 45,483개(99.0%)는 dl_idx가 K-코드보다 1 작고, 나머지 450개는 전혀 다른 값이다.
    - Kaggle은 763개 모두 dl_idx가 K-코드와 같다. 대회 측이 dl_idx를 K-코드로 고친 것으로 보인다.
- 원본에는 학습 전에 걸러야 할 파일이 섞여 있다: -오류 폴더 9개, 읽을 수 없는 JSON 1개, copy 중복 24개, macOS ._ 파일 1개

**결론**
- 키 구성이 같으므로 키는 그대로 두고 값만 고친다. 고친 JSON은 train과 같은 `load_annotations`로 읽는다.
    - `iscrowd`가 없는 JSON 1개는 변환할 때 0으로 채운다.
- AI Hub 라벨의 category_id는 dl_idx가 아니라 drug_N(K-코드)에서 만든다.
    - dl_idx를 그대로 쓰면 모든 클래스가 하나씩 어긋나고, 에러 없이 점수만 0이 된다.
- image id와 annotation id는 대회 측 가공(DATA.md)처럼 새로 매긴다.
- 이 처리는 공용 모듈의 `read_aihub_label_zip`과 `convert_aihub_to_kaggle`에 넣었다.

- 같은 알약 K-001900(보령부스파정 5mg)을 같은 각도(70°)에서 찍은 Kaggle JSON 하나와 AI Hub JSON 하나를 키마다 나란히 놓는다.
    - Kaggle 쪽은 0번 `train_records 예시` 셀의 이미지(`K-001900-016548-019607-029451_..._70_...`)에 있는 K-001900의 JSON이다.
    - AI Hub 쪽은 허용 zip에서 같은 K-코드, 같은 각도의 JSON 중 파일명 순서로 첫 번째이다.
    - 같은 알약이므로 알약 정보는 같아야 하고, 사진이 달라서 생기는 값과 대회 측이 고친 값만 달라야 한다.
- 값이 다른 키는 `비교` 열에 **다름**으로 표시한다. 긴 값은 앞 30자만 보인다.

In [ ]:
EXAMPLE_CODE = 'K-001900'
kaggle_json_path = (TRAIN_ANN_DIR / 'K-001900-016548-019607-029451_json' / EXAMPLE_CODE
                    / 'K-001900-016548-019607-029451_0_2_0_2_70_000_200.json')
kaggle_json = json.loads(kaggle_json_path.read_text(encoding='utf-8'))


def is_example(name):
    """같은 K-코드 폴더에 있는 70° JSON인지 본다. copy, ._ 파일과 -오류 폴더는 뺀다."""
    parts = unicodedata.normalize('NFC', name).split('/')
    return (len(parts) >= 2 and parts[-2] == EXAMPLE_CODE and '_70_' in parts[-1] and parts[-1].endswith('.json')
            and not parts[-1].startswith('._') and ' copy' not in parts[-1])


aihub_json = None
for path in label_zips:
    with zipfile.ZipFile(path) as zf:
        names = sorted(n for n in zf.namelist() if is_example(n))
        if names:
            aihub_json_name = unicodedata.normalize('NFC', names[0]).split('/')[-1]
            aihub_json = json.loads(zf.read(names[0]))
            break
assert aihub_json is not None, f'허용 zip에 {EXAMPLE_CODE}의 70° JSON이 없다'


def short(value, n=30):
    """표에 넣을 값을 앞 n자만 남긴다. 빈 문자열은 (빈 값)으로 쓴다."""
    text = str(value).replace('|', '/')
    if text == '':
        return '(빈 값)'
    return text if len(text) <= n else text[:n] + '…'


rows = [['파일', '-', short(kaggle_json_path.name, 40), short(aihub_json_name, 40), '-'],
        ['최상위', '(키 목록)', ', '.join(kaggle_json), ', '.join(aihub_json),
         '같음' if list(kaggle_json) == list(aihub_json) else '**다름**']]
n_keys = n_diff = 0
for part in ['images', 'annotations', 'categories']:
    for key, kaggle_value in kaggle_json[part][0].items():
        aihub_value = aihub_json[part][0].get(key, '(키 없음)')
        n_keys += 1
        n_diff += kaggle_value != aihub_value
        rows.append([part, key, short(kaggle_value), short(aihub_value), '같음' if kaggle_value == aihub_value else '**다름**'])
rows.append([f'**값이 다른 키 {n_diff}/{n_keys}개**', '', '', '', ''])
show_table(['위치', '키', 'Kaggle train', 'AI Hub 조합 라벨 (허용 zip)', '비교'], rows)

**인사이트**
- 같은 알약의 두 JSON은 최상위 키 4개와 하위 키 61개(images 50, annotations 8, categories 3)의 이름과 순서가 같다.
- 값이 다른 키는 61개 중 11개이다.
- 4개는 사진이 달라서 다르다.
    - `file_name`, `imgfile`(사진 파일명), `bbox`, `area`(알약 위치와 넓이)
- 7개는 id와 클래스 표기이다. AI Hub는 1이나 "Drug"이고, Kaggle은 대회 측이 고친 값이다.
    - images `id`(34, AI Hub 1), annotations `id`(133, AI Hub 1), `image_id`(34, AI Hub 1)
    - annotations `category_id`, categories `id`: Kaggle은 K-코드 숫자 1900, AI Hub는 1
    - images `dl_idx`: Kaggle 1900, AI Hub 1899 (K-코드보다 1 작음, 6-2)
    - categories `name`: Kaggle은 제품명(보령부스파정 5mg), AI Hub는 Drug
- 알약 정보(제품명, 성분, 회사, 각인 BSP / 5, 색, 모양, 크기 8×4.5×2.5mm)와 촬영 조건(배경, 조명, 각도)은 모두 같다.

**결론**
- 변환할 때는 7개 값만 고치고, 나머지 키는 원본 값을 그대로 복사한다.
    - id 3개는 새로 매긴다.
    - `category_id`, categories `id`, `dl_idx`는 drug_N의 숫자로 바꾼다.
    - categories `name`은 `dl_name`(제품명)으로 바꾼다.
    - 공용 모듈의 `convert_aihub_to_kaggle`이 이 규칙대로 바꾼다. area는 2-2처럼 너비×높이로 다시 계산한다.
- `bbox`는 Kaggle과 같은 [x, y, w, h] 픽셀 형식으로 보고 변환하지 않는다. 이미지를 받은 뒤(Phase B) box를 겹쳐 그려 확인한다.

### 6-3. 출처 확인
- `read_aihub_label_zip`(공용 모듈)으로 허용 zip 8개를 이미지 단위로 읽는다.
    - 이 함수는 6-2에서 찾은 차이를 처리한다: category_id를 drug_N에서 만들고, copy와 `._` 파일을 버리고, 오류가 있는 이미지를 표시한다.
- train 이미지, train 조합, 라벨이 누락된 8장(2-1)이 허용 zip에 있는지 센다.
    - 없다면 대회 데이터는 금지된 `TS_2_조합`에서 나왔다는 뜻이고, 누락 라벨은 AI Hub에서 복구할 수 없다.

In [ ]:
aihub_records = {}
for path in label_zips:
    aihub_records.update(read_aihub_label_zip(path))
aihub_combos = {combo_name(fn) for fn in aihub_records}
show_table(['확인 항목', '허용 zip에 있는 수'], [
    ['train 이미지', f'{len(set(train_files) & set(aihub_records))}/{len(train_files)}'],
    ['train 조합', f'{len(set(train_combos) & aihub_combos)}/{len(train_combos)}'],
    ['라벨이 누락된 train 이미지 (2-1)', f'{len(set(missing_files) & set(aihub_records))}/{len(missing_files)}'],
    ['허용 zip의 이미지 / 조합', f'{len(aihub_records):,}장 / {len(aihub_combos):,}개'],
])

**인사이트**
- train 이미지 232장과 train 조합 114개 모두 허용 zip에 없다.
- 라벨이 누락된 8장도 허용 zip에 없다.
- 허용 zip에는 12,003장, 4,001조합이 있다. 조합마다 3장이다.

**결론**
- 대회 데이터(train, test)는 금지된 `TS_2_조합`에서 나온 것으로 본다.
- 누락 라벨은 AI Hub에서 복구할 수 없으므로, 그 8장은 2-1에서 사진으로 그린 box를 쓴다. (2-5)
- 허용 zip은 대회 train과 겹치지 않는다. test와 겹치는지는 이미지를 받은 뒤(Phase B) 확인한다.

### 6-4. 클래스 커버리지
- 허용 zip의 이미지 중 오류가 없는 이미지만 쓴다.
    - 오류: "-오류" 폴더, 읽을 수 없는 JSON, 비정상 bbox, 파일명보다 적은 라벨 (`read_aihub_label_zip` 설명 참고)
    - 라벨이 빠진 알약이 있는 이미지를 학습에 넣으면, 그 알약을 배경으로 배운다.
- 왼쪽: train 56개 클래스의 객체 수를 대회 원본(2-5 정제 라벨)과 허용 zip에서 비교한다. (로그 눈금)
- 오른쪽: train에 없는 클래스의 허용 zip 객체 수를 본다. 이 클래스들이 test 전용 클래스 후보이다.

In [ ]:
aihub_clean = {fn: r for fn, r in aihub_records.items() if not r['errors']}
aihub_error_count = collections.Counter(e for r in aihub_records.values() for e in r['errors'])
aihub_object_count = collections.Counter(a['category_id'] for r in aihub_clean.values() for a in r['annotations'])
aihub_classes = set(aihub_object_count)
new_classes = sorted(aihub_classes - set(train_classes), key=lambda c: -aihub_object_count[c])
absent_train = sorted(set(train_classes) - aihub_classes)

fig, axes = plt.subplots(1, 2, figsize=(20, 6), layout='constrained', gridspec_kw={'width_ratios': [1, 1.1]})
order = sorted(train_classes, key=lambda c: (-clean_object_count[c], c))
x = np.arange(len(order))
axes[0].bar(x - 0.2, [clean_object_count[c] for c in order], width=0.4, label='대회 원본 train (2-5 정제 라벨)')
axes[0].bar(x + 0.2, [aihub_object_count[c] for c in order], width=0.4, label='허용 zip (오류 이미지 제외)')
axes[0].set_yscale('log')
axes[0].set_xticks(x, [f'K-{c:06d}' for c in order], rotation=90, fontsize=7)
axes[0].set_ylabel('객체 수 (개, 로그 눈금)')
axes[0].set_title(f'train 클래스 {len(train_classes)}개: 원본과 허용 zip의 객체 수\n'
                  f'허용 zip에 없는 train 클래스: {", ".join(f"K-{c:06d}" for c in absent_train)} (막대 높이 0)')
axes[0].legend()
bars = axes[1].bar([f'K-{c:06d}' for c in new_classes], [aihub_object_count[c] for c in new_classes], color='tab:green')
axes[1].tick_params(axis='x', rotation=90, labelsize=7)
axes[1].set_ylabel('객체 수 (개)')
axes[1].set_title(f'train에 없는 클래스 {len(new_classes)}개의 허용 zip 객체 수 (test 전용 클래스 후보)\n'
                  f'최소 {min(aihub_object_count[c] for c in new_classes)}개, 중앙값 {int(np.median([aihub_object_count[c] for c in new_classes]))}개')
fig.suptitle(f'허용 zip 오류 없는 이미지 {len(aihub_clean):,}장의 클래스 커버리지 '
             f'(오류 이미지 {len(aihub_records) - len(aihub_clean)}장 제외: '
             + ', '.join(f'{k} {v}' for k, v in aihub_error_count.most_common()) + ')')
save_fig(fig, '6-4_coverage')
plt.show()

**인사이트**
- 오류 없는 이미지는 11,840장이다. 163장(1.4%)은 라벨 누락, 비정상 bbox, -오류 폴더 때문에 뺀다.
- train 56개 클래스 중 54개가 허용 zip에 있고, 객체가 약 130~1,300개로 원본(3~37개)보다 훨씬 많다.
    - 원본에서 1조합에만 나오는 17개 클래스(객체 3개)도 허용 zip에는 수백 개씩 있다.
- K-003351과 K-003483은 허용 zip에 없다. 원본에는 각각 157개, 45개가 있다.
- train에 없는 클래스가 62개 있고, 객체 수는 90~640개(중앙값 369개)이다.

**결론**
- 허용 zip으로 train 희소 클래스와 test 전용 후보를 함께 채울 수 있다.
- 학습 클래스는 train 56개와 허용 zip의 62개를 합친 118개로 한다.
    - 조합 사진에는 알약 여러 개가 함께 있어서, 이미지를 넣으면 그 안의 알약을 모두 라벨과 함께 넣어야 한다.
    - COCO 방식 mAP는 test에 정답이 없는 클래스를 평균에서 빼므로, test에 없는 클래스가 섞여도 손해가 작다.
- K-003351과 K-003483은 원본 데이터만으로 학습한다. 객체가 157개, 45개라 부족하지 않다.

### 6-5. test에서 읽은 각인 대조
- 5-3 기록표의 각인을 허용 zip 라벨의 각인 필드(print_front, print_back)에서 찾는다.
    - 띄어쓰기와 대소문자는 무시하고, 필드 값 전체가 같은 경우를 먼저 찾는다.
    - 같은 값이 없으면 필드 값에 각인이 들어 있는 경우(부분 일치)를 찾는다.
    - AI Hub는 분할선을 "분할선"이라는 글자로 적으므로, 기록표의 `|`를 바꿔서 찾는다. (K|50 → `K분할선50`)

In [ ]:
def normalize(text):
    """띄어쓰기와 대소문자를 무시하고 비교하도록 각인 글자를 정리한다."""
    return str(text or '').replace(' ', '').lower()


aihub_pill_info = {}  # K-코드 → 그 알약 JSON의 이미지 메타데이터
for record in aihub_clean.values():
    for ann in record['annotations']:
        aihub_pill_info.setdefault(ann['category_id'], ann['image'])
filled = sum(bool(im['print_front'] or im['print_back']) for im in aihub_pill_info.values())
imprint_fields = {c: (normalize(im['print_front']), normalize(im['print_back'])) for c, im in aihub_pill_info.items()}

files_by_imprint = collections.defaultdict(set)  # 기록표의 각인 → test 파일 번호
look_by_imprint = {}  # 기록표의 각인 → test에서 본 색과 모양
imprint_matches = {}  # 기록표의 각인 → 허용 zip에서 찾은 K-코드
for row in review_rows:
    if row['imprint'] not in ('', '?'):  # 각인을 읽지 못한 알약은 글자로 대조할 수 없어 뺀다
        files_by_imprint[row['imprint']].add(int(row['test_file'].split('.')[0]))
        look_by_imprint.setdefault(row['imprint'], f"{row['color']} {row['shape']}")

rows = []
for seen, numbers in sorted(files_by_imprint.items(), key=lambda kv: (-len(kv[1]), kv[0])):
    query = normalize(seen.replace('|', '분할선'))
    match, hits = '같음', [c for c, fields in imprint_fields.items() if query in fields]
    if not hits and len(query) >= 2:
        match, hits = '부분 일치', [c for c, fields in imprint_fields.items() if any(query in f for f in fields)]
    imprint_matches[seen] = hits
    shown = ', '.join(map(str, sorted(numbers)[:6])) + (' …' if len(numbers) > 6 else '')
    for c in hits:
        info = aihub_pill_info[c]
        rows.append([seen, look_by_imprint[seen], shown, f'K-{c:06d}', info['dl_name'], f"{info['color_class1']} {info['drug_shape']}",
                     match, '있음' if c in train_classes else '없음', aihub_object_count[c]])
    if not hits:
        rows.append([seen, look_by_imprint[seen], shown, '찾지 못함', '-', '-', '-', '-', '-'])
rows.append([f'**각인 필드가 있는 클래스 {filled}/{len(aihub_pill_info)}**', '', '', '', '', '', '', '', ''])
show_table(['test에서 읽은 각인', 'test 색, 모양', 'test 파일 번호', '허용 zip K-코드', '제품명', '허용 zip 색, 모양', '일치',
            'train에 있음', '허용 zip 객체 수'], rows)

**인사이트**
- 허용 zip 116개 클래스 중 114개는 각인 필드가 채워져 있다.
- test에서 읽은 각인 19종 중 17종이 허용 zip의 각인 필드와 같다. 모두 train에 없는 클래스이다.
    - 213은 부분 일치이지만, 같은 K-018110(란스톤엘에프디티정 30mg)의 앞면 `마크213`, 뒷면 `30`과 맞는다. test의 213 알약 뒷면도 30이다.
    - 그래서 test 전용 알약 18종 중 17종의 K-코드를 찾았다. 허용 zip 객체는 285~459개이다.
- Hanmi(노란 타원형 정제, test 28장)는 찾지 못했다.
    - 부분 일치한 K-015280은 한미탐스캡슐(주황 장방형 캡슐, 각인 Hanmi 0.2)이라 test의 알약과 다르다.
- 색 표기가 test와 다른 경우가 있다. (C77, B-F: 허용 zip은 갈색, test에서는 빨강으로 적음)
    - 각인과 모양이 맞으므로 같은 알약으로 본다. 색 이름은 사람마다 다르게 붙인다.
- 사람이 기록표에 메모한 제품명 추정(비타비백정, 조인스정 200mg, 삐콤씨에프정, 란스톤엘에프디티정 30mg, 쎄로켈정 100mg)도 이 표의 제품명과 같다.

**결론**
- test 전용 클래스 17개(K-005094, K-005886, K-006192, K-006563, K-010221, K-010224, K-012420, K-018110, K-021026, K-022627, K-023203, K-023223, K-027653, K-029871, K-033878, K-044199, K-004378)는 모두 허용 zip의 train에 없는 클래스 62개 안에 있다. 그래서 62개를 모두 학습 클래스에 넣는다. (6-4)
- Hanmi 정제는 허용 조합 zip에 없는 클래스로 보고, 단일 데이터에서 찾는다. (Phase B)

### 6-6. 보충 목록과 용량
- 클래스마다 목표 객체 수(2-5에서 정제한 대회 원본 232장 포함)를 정하고, 목표를 채우는 데 필요한 이미지만 고른다.
    - 대상 클래스: 대회 원본 train 클래스와 허용 zip의 모든 클래스
        - 조합 사진에는 알약 3~4개가 함께 있어서, 이미지를 넣으면 그 안의 모든 알약을 라벨과 함께 넣어야 한다.
    - zip 고르기: 아직 목표에 못 미친 객체를 가장 많이 채우는 zip부터 고른다.
    - 이미지 고르기: 고른 zip 안에서, 목표에 못 미친 클래스의 객체를 가장 많이 가진 이미지부터 고른다.
- 남길 용량은 대회 train PNG 평균 크기로 추정한다. 받는 동안에는 zip 하나당 약 6GB가 더 필요하다. (tar와 part 파일)

In [ ]:
TARGETS = [50, 100, 200, 400]
ZIP_GB = 3  # 조합 이미지 zip 하나의 용량 (aihubshell 표기)
avg_png_mb = np.mean([(TRAIN_IMG_DIR / fn).stat().st_size for fn in train_files]) / 2**20

images_by_zip = collections.defaultdict(list)
for fn, record in aihub_clean.items():
    images_by_zip[record['zip']].append(fn)
image_classes = {fn: [a['category_id'] for a in r['annotations']] for fn, r in aihub_clean.items()}
all_classes = sorted(set(train_classes) | aihub_classes)


def plan_supplement(target):
    """목표 객체 수를 채우는 zip 순서와 이미지 목록을 고른다."""
    count = collections.Counter(clean_object_count)  # 보충 전 객체 수는 2-5의 정제 라벨로 센다
    chosen_zips, chosen_images = [], []
    remaining = dict(images_by_zip)
    while remaining:
        def gain(z):
            need = collections.Counter({c: max(0, target - count[c]) for c in all_classes})
            got = collections.Counter(c for fn in remaining[z] for c in image_classes[fn])
            return sum(min(need[c], got[c]) for c in all_classes)
        best = max(sorted(remaining), key=gain)
        if gain(best) == 0:
            break
        chosen_zips.append(best)
        # zip 안에서 아직 필요한 클래스를 많이 가진 이미지부터, 여전히 필요한 클래스가 있으면 넣는다
        pool = remaining.pop(best)
        first_score = {fn: sum(count[c] < target for c in image_classes[fn]) for fn in pool}
        for fn in sorted(pool, key=lambda f: (-first_score[f], f)):
            if any(count[c] < target for c in image_classes[fn]):
                chosen_images.append(fn)
                count.update(image_classes[fn])
    short = sum(count[c] < target for c in all_classes)
    return chosen_zips, chosen_images, short


plans = {t: plan_supplement(t) for t in TARGETS}
rows = []
for t, (zips, images, short) in plans.items():
    rows.append([t, len(zips), f'{len(zips) * ZIP_GB}GB', f'{len(images):,}장',
                 f'{len(images) * avg_png_mb / 1024:.1f}GB', f'{short}/{len(all_classes)}'])
show_table(['클래스별 목표 객체 수', '받을 zip 수', '받는 용량 (합계)', '꺼낼 이미지', '남길 용량 추정', '목표에 못 미친 클래스'], rows)

**인사이트**
- 목표 50은 zip 7개를 받아 이미지 2,827장(약 4.6GB)을 남긴다.
- 목표 100은 zip 8개를 모두 받아 5,192장(약 8.5GB)을 남긴다.
- 목표 200은 8,481장(약 13.8GB), 목표 400은 11,602장(약 18.9GB)이다.
- 목표에 못 미치는 클래스는 목표 100에서 2개, 200에서 16개, 400에서 73개이다.
    - 목표 100에서 못 미치는 2개는, 허용 zip에 없는 K-003483(원본 45개)과 허용 zip 객체가 90개뿐인 K-020753이다.
- 남길 용량은 대회 PNG 평균 크기로 추정했다. AI Hub zip 용량으로 계산하면(3GB ÷ 1,500장, 약 2MB) 약 20% 더 크다.

**결론**
- 클래스별 목표 객체 수는 100개로 정한다.
    - 남길 용량 8.5GB(넉넉히 잡아 약 10GB)에, 받는 동안 zip 하나당 더 쓰는 약 6GB를 더해도 남은 디스크 23GiB 안에 들어온다.
    - 목표 200은 남길 용량만 13.8~16.6GB라, YOLO 형식으로 바꾸는 과정까지 생각하면 디스크가 모자란다.
    - 목표 100이면 원본에서 객체가 3개뿐인 클래스도 100개 이상이 된다.
- YOLO 형식으로 바꿀 때는 이미지를 복사하지 않고 링크(symlink)로 만들어 디스크를 아낀다.

- 6-6에서 정한 목표 객체 수로 보충 목록을 `data/external/aihub_targets.csv`에 저장하고, zip별로 받을 파일과 꺼낼 이미지 수를 정리한다.
    - 라벨 zip(TL_n, VL_1)과 같은 번호의 이미지 zip(TS_n, VS_1)에 같은 이미지가 들어 있다.

In [ ]:
SELECTED_TARGET = 100
IMAGE_ZIP = {  # 라벨 zip → (이미지 zip, filekey)
    'TL_1_조합.zip': ('TS_1_조합.zip', 66154), 'TL_3_조합.zip': ('TS_3_조합.zip', 66156),
    'TL_4_조합.zip': ('TS_4_조합.zip', 66157), 'TL_5_조합.zip': ('TS_5_조합.zip', 66158),
    'TL_6_조합.zip': ('TS_6_조합.zip', 66159), 'TL_7_조합.zip': ('TS_7_조합.zip', 66160),
    'TL_8_조합.zip': ('TS_8_조합.zip', 66161), 'VL_1_조합.zip': ('VS_1_조합.zip', 66254),
}
zips, images, _ = plans[SELECTED_TARGET]
target_csv = PROJECT_ROOT / 'data' / 'external' / 'aihub_targets.csv'
with open(target_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['label_zip', 'image_zip', 'filekey', 'file_name'])
    for fn in sorted(images, key=lambda n: (zips.index(aihub_clean[n]['zip']), n)):
        label_zip = aihub_clean[fn]['zip']
        writer.writerow([label_zip, *IMAGE_ZIP[label_zip], fn])

per_zip = collections.Counter(aihub_clean[fn]['zip'] for fn in images)
show_table(['받는 순서', '이미지 zip', 'filekey', '꺼낼 이미지 수', '남길 용량 추정'], [
    [i + 1, IMAGE_ZIP[z][0], IMAGE_ZIP[z][1], f'{per_zip[z]:,}장', f'{per_zip[z] * avg_png_mb / 1024:.1f}GB']
    for i, z in enumerate(zips)
] + [['**합계**', '', '', f'**{len(images):,}장**', f'**{len(images) * avg_png_mb / 1024:.1f}GB**']])

**인사이트**
- 목표 100이면 zip 8개를 모두 받고, 이미지 5,192장(약 8.5GB)을 꺼낸다.
- 앞의 두 zip(TS_6, TS_4)에서 2,805장(54%)을 꺼낸다.
- 뒤로 갈수록 꺼낼 이미지가 줄어, 마지막 VS_1은 3GB를 받아 86장만 꺼낸다.
    - 앞 zip들로 대부분 클래스가 목표를 채우고, 남은 소수 클래스만 뒤 zip에서 채우기 때문이다.

**결론**
- 표의 순서대로 zip을 하나씩 받고, `data/external/aihub_targets.csv`에 있는 이미지만 꺼낸 뒤 zip을 지운다.
- 받기 전에는 매번 남은 디스크가 12GB 이상인지 확인한다.

### 6-7. 보충 전후 클래스 분포
- 6-6에서 고른 보충 이미지의 라벨로, 보충 전후의 클래스별 객체 수와 등장 조합 수를 비교한다.
    - 라벨 zip에 bbox와 K-코드가 있으므로, 이미지를 받기 전에 계산할 수 있다.
    - 보충 전은 2-5의 정제 라벨(대회 원본 train 232장)이다.
- 등장 조합 수가 2개 이상이어야 조합 단위 분할에서 train과 val 양쪽에 둘 수 있다. (4-1)
- 목적: 보충 후 조합 단위 분할에서 모든 클래스를 val에 넣을 수 있는지 확인하고, 클래스 불균형 대응 방법을 정한다.

In [ ]:
supplement_records = {fn: aihub_clean[fn] for fn in plans[SELECTED_TARGET][1]}


def class_stats(records):
    """클래스별 객체 수와 등장 조합 수를 센다."""
    objects, combos = collections.Counter(), collections.defaultdict(set)
    for fn, record in records.items():
        for ann in record['annotations']:
            objects[ann['category_id']] += 1
            combos[ann['category_id']].add(combo_name(fn))
    return objects, {c: len(s) for c, s in combos.items()}


def class_list(objects, keep):
    """조건에 맞는 클래스를 객체 수 순서로 적는다. 6개 이상이면 개수만 적는다."""
    picked = sorted((n, c) for c, n in objects.items() if keep(n))
    if len(picked) > 5:
        return f'{len(picked)}개'
    return ', '.join(f'K-{c:06d} {n}개' for n, c in picked) or '없음'


BINS = [(0, 100), (100, 200), (200, 400), (400, float('inf'))]  # 객체 수 구간
columns = []
for records in [clean_records, {**clean_records, **supplement_records}]:
    objects, combos = class_stats(records)
    values = sorted(objects.values())
    total = sum(values)
    top_code, top_n = objects.most_common(1)[0]
    columns.append([
        f'{len(records):,}장',
        f'{total:,}개',
        f'{len(objects)}개',
        f'{values[0]} / {np.median(values):g} / {values[-1]}',
        f'{values[-1] / values[0]:.1f}배',
        f'K-{top_code:06d} {top_n}개 ({top_n / total:.1%})',
        f'{objects[3351]}개 ({objects[3351] / total:.1%})',
        ' / '.join(str(sum(lo <= n < hi for n in values)) for lo, hi in BINS),
        class_list(objects, lambda n: n < 100),
        class_list(objects, lambda n: n >= 400),
        f'{sum(n == 1 for n in combos.values())}개',
        f'{min(combos.values())} / {np.median(list(combos.values())):g}',
    ])

row_names = ['이미지 수', '객체 수', '클래스 수', '클래스별 객체 수 (최소 / 중앙값 / 최대)', '최대 ÷ 최소',
             '객체가 가장 많은 클래스 (비중)', 'K-003351 (원본에서 가장 많던 클래스)',
             '객체 수 구간별 클래스 수 (100 미만 / 100~199 / 200~399 / 400 이상)',
             '객체 100개 미만 클래스', '객체 400개 이상 클래스',
             '1조합에만 나오는 클래스 (조합 단위 분할에서 val에 넣을 수 없음)', '클래스별 등장 조합 수 (최소 / 중앙값)']
show_table(['항목', '보충 전 (대회 원본, 2-5 정제 라벨)', f'보충 후 (+ 허용 zip {len(supplement_records):,}장)'],
           [[name, before, after] for name, before, after in zip(row_names, *columns)])

**인사이트**
- 보충 후에는 이미지 5,424장, 객체 20,198개, 클래스 118개이다. (보충 전 232장, 771개, 56개)
- 클래스별 객체 수의 최대÷최소가 52.3배에서 17.0배로 줄어든다.
    - 118개 중 94개가 100~199개 구간에 든다. 중앙값은 145개이다.
    - 원본에서 20.4%를 차지하던 K-003351은 157개 그대로이고, 비중은 0.8%로 줄어든다.
- 목표 100에 못 미치는 클래스는 2개이다.
    - K-003483(45개)은 허용 zip에 없어 원본 객체만 있다.
    - K-020753(90개)은 허용 zip에 있는 객체를 모두 넣은 값이다.
- 4개 클래스는 400개를 넘는다. (K-016232 763개, K-011220 619개, K-016548 604개, K-016551 540개)
    - 부족한 클래스를 채우려고 고른 사진에 이 알약들이 함께 찍혀 있어서 따라 들어온 객체이다.
- 1조합에만 나오는 클래스는 17개에서 0개가 된다. 가장 적은 클래스도 15조합에 나온다.

**결론**
- 보충 후에는 모든 클래스를 조합 단위로 train과 val 양쪽에 둘 수 있다.
    - val 20%면 가장 적은 클래스도 기댓값으로 3조합이 val에 들어간다.
    - 분할은 이미지를 받은 뒤(Phase B) `make_group_split`으로 만들고, 양쪽에 있는 클래스 수를 확인한다.
- 기준선은 클래스 가중치나 반복 샘플링 없이 학습한다.
    - 클래스 94개가 100~199개 구간에 모여 있고, 가장 많은 클래스도 전체 객체의 3.8%이다.
- 객체가 적은 K-003483(45개), K-020753(90개)은 val의 클래스별 AP로 따로 본다.
    - AP가 다른 클래스보다 낮으면, 이 클래스가 든 이미지를 반복해서 넣는 실험을 한다.
- 400개를 넘는 4개 클래스는 줄이지 않는다. 사진을 빼면 함께 찍힌 다른 클래스도 같이 빠진다.

- 6-7 표의 보충 후 클래스별 객체 수를 K-코드마다 막대 하나로 그린다.
    - 클래스를 세 그룹으로 나눠 색을 다르게 하고, 그룹 안에서는 객체 수 순서로 놓는다.
        - 원본 train 클래스: 대회 원본 train에 있는 56개
        - test 전용 클래스: train에 없고, 6-5에서 test 사진의 각인으로 K-코드를 찾은 17개
        - 허용 zip으로 추가되는 클래스: train에 없고 test에서 확인하지 못한 45개 (6-4에서 학습 클래스에 넣기로 함)
    - 원본 train 클래스 막대 아래쪽의 짙은 남색은 보충 전에 있던 대회 원본 객체(2-5 정제 라벨)이다.
- 6-5에서 Hanmi와 부분 일치한 K-015280(한미탐스캡슐)은 test의 알약과 다른 클래스라서 test 전용 클래스에 넣지 않는다.

In [ ]:
WRONG_MATCHES = {('Hanmi', 15280)}  # 6-5에서 부분 일치했지만 test의 알약과 다른 클래스 (6-5 인사이트 참고)
test_only_classes = {c for seen, hits in imprint_matches.items() for c in hits if (seen, c) not in WRONG_MATCHES}
assert test_only_classes <= set(new_classes), 'test 전용 클래스는 train에 없는 허용 zip 클래스여야 한다'
after_objects, _ = class_stats({**clean_records, **supplement_records})
groups = [  # (그룹 이름, 클래스, 막대 색)
    ('원본 train 클래스', set(train_classes), 'tab:blue'),
    ('test 전용 클래스 (6-5에서 각인으로 찾음)', test_only_classes, 'tab:orange'),
    ('허용 zip으로 추가되는 클래스', set(new_classes) - test_only_classes, 'tab:green'),
]

fig, ax = plt.subplots(figsize=(24, 7), layout='constrained')
start, ticks = 0, []
top = max(after_objects.values())
for name, classes, color in groups:
    order = sorted(classes, key=lambda c: (-after_objects[c], c))
    x = np.arange(start, start + len(order))
    before = np.array([clean_object_count[c] for c in order])
    after = np.array([after_objects[c] for c in order])
    if before.any():
        ax.bar(x, before, color='#08306b', label='보충 전에 있던 대회 원본 객체 (2-5 정제 라벨)')
    ax.bar(x, after - before, bottom=before, color=color,
           label=f'{name} {len(order)}개 (객체 {after.min()}~{after.max()}개, 중앙값 {np.median(after):g}개)')
    for xi, b, a in zip(x, before, after):
        if a < SELECTED_TARGET or a >= 400 or a == b:  # 목표에 못 미친 클래스, 400개 넘는 클래스, 보충이 없는 클래스
            ax.text(xi, a + 8, str(a), ha='center', va='bottom', rotation=90, fontsize=8)
    if start:
        ax.axvline(start - 0.5, color='gray', linewidth=0.8, linestyle=':')
    ticks += order
    start += len(order)
ax.axhline(SELECTED_TARGET, color='black', linewidth=1, linestyle='--', label=f'6-6 목표 객체 수 {SELECTED_TARGET}개')
ax.set_xticks(np.arange(len(ticks)), [f'K-{c:06d}' for c in ticks], rotation=90, fontsize=7)
ax.set_xlim(-1, len(ticks))
ax.set_ylim(0, top * 1.1)
ax.set_ylabel('보충 후 객체 수 (개)')
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[1:] + handles[:1], labels[1:] + labels[:1], loc='upper right', fontsize=10)  # 목표 선을 범례 맨 아래로
ax.set_title(f'보충 후 클래스 {len(ticks)}개의 객체 수 (대회 원본 {len(clean_records)}장 + 허용 zip {len(supplement_records):,}장)\n'
             '그룹 안에서 객체 수 순서로 정렬, 막대 위 숫자 = 목표 미달·400개 이상·보충이 없는 클래스의 객체 수')
save_fig(fig, '6-7b_class_objects_after')
plt.show()

**인사이트**
- 세 그룹의 객체 수 중앙값은 비슷하다. (원본 train 클래스 152.5개, test 전용 클래스 133개, 허용 zip으로 추가되는 클래스 143개)
- test 전용 클래스 17개는 모두 목표 100개 이상이다. (100~233개)
    - 객체는 모두 허용 zip에서 온 것이다. 대회 원본에는 이 클래스의 객체가 없다.
- 원본 train 클래스의 보충 후 객체 10,511개 중 대회 원본 객체는 771개(7.3%)이다.
    - 클래스별 원본 객체 비중의 중앙값은 5.3%이다.
    - K-003351(157개)과 K-003483(45개)은 허용 zip에 없어서 막대 전체가 대회 원본 객체이다.
- 400개를 넘는 4개 클래스 중 3개(K-016232, K-016548, K-016551)는 원본 train 클래스이고, K-011220(619개)은 허용 zip으로 추가되는 클래스이다.
- 허용 zip으로 추가되는 클래스 45개 중 13개는 객체가 목표와 같은 100개이다.
    - 6-6은 목표에 못 미친 클래스가 든 사진만 고르므로, 이 클래스들은 목표를 채운 뒤 더 늘지 않았다.

**결론**
- 그룹에 따라 클래스 가중치를 다르게 주지 않는다. 세 그룹의 객체 수가 같은 범위에 있다.
- val의 클래스별 AP를 세 그룹으로 묶어 그룹 평균도 본다.
    - test 전용 클래스는 대회 원본 객체 없이 허용 zip 사진으로만 배운다. 그리고 train에 없는 알약은 test 사진의 39.8%에 나온다. (5-3)
    - test 전용 그룹의 평균 AP가 다른 그룹보다 낮으면, 이 17개 클래스만 목표 객체 수를 200개로 올려 보충하고 비교한다.

### 6-8 이후: 이미지를 받은 뒤 진행할 분석 (Phase B)
- 6-8. 중복 검사: 받은 이미지와 대회 train, test 사이에 파일 해시와 축소 이미지 기준으로 같은 사진이 있는지 확인한다.
    - 겹쳐서 빼는 이미지가 있으면 6-6의 보충 목록과 6-7의 분포를 다시 계산한다.
- 6-9. test 전용 클래스 확정: 6-5에서 찾은 K-코드의 알약을 받은 이미지에서 잘라, 5-3의 test 알약과 같은 알약인지 눈으로 대조한다. 각인을 읽지 못한 알약(`?`)도 함께 대조한다.
- 6-10. Kaggle 형식 변환과 미리보기: `convert_aihub_to_kaggle`로 `data/external/aihub_converted/`에 data/raw와 같은 구조(`train_annotations/`, `train_images/`)로 저장하고, box를 겹쳐 그려 확인한다.
- 6-11. (선택) 조합에 없는 클래스: 6-5의 Hanmi 정제처럼 허용 조합 zip에 없는 클래스는 단일 라벨 zip(합계 2.8GB)의 각인 필드로 K-코드와 들어 있는 이미지 zip을 찾는다. 이미지 zip은 로컬 디스크로 받을 수 없으므로 별도 환경을 정한다.
- 이미지 zip을 받을 때 지킬 점
    - `script/download_aihub.py`로 받는다. 이미지 zip은 `data/external/aihub/train_images/`에 생긴다.
        - `aihubshell`의 병합 단계는 macOS 기본 bash(3.2)에서 한글 파일명을 찾지 못해 빈 zip을 만들고 part 파일까지 지운다. 스크립트는 같은 API로 받고 part 병합을 직접 한다.
    - zip은 한 번에 하나씩 받고, `aihub_targets.csv`에 있는 이미지만 `aihub_converted/train_images/`로 꺼낸 뒤 바로 지운다.

## 7. 분할 만들기 (Phase B)
- 원본과 보충 데이터를 합친 뒤 `make_group_split`(공용 모듈)으로 조합 단위로 나눈다.
    - 조합 순서를 200번 섞어 보고, train과 val 양쪽에 있는 클래스 수가 가장 많은 분할을 고른다.
    - seed가 같으면 누구나 같은 분할을 얻는다.
- 이 섹션은 보충 이미지를 받은 뒤 작성한다.

## 8. 평가 기준선
- 목적: 로컬 평가 코드를 검증하고, 제출 형식과 라벨 정밀도의 기준을 정한다.

### 8-1. mAP 계산 검증
- 대회 지표 mAP@[0.75:0.95]를 pycocotools(COCO 공식 평가 코드)로 계산한다.
    - AP(Average Precision)는 한 클래스에서 점수가 높은 예측부터 차례로 볼 때의 정밀도-재현율 곡선 아래 넓이이다.
    - mAP@[0.75:0.95]는 IoU 기준 0.75, 0.80, 0.85, 0.90, 0.95마다 클래스별 AP를 구해 모두 평균한 값이다.
- 정답은 2-5의 정제 라벨(이미지 232장, 객체 771개)이다. 로컬 val도 정제 라벨로 평가하기 때문이다.
- 정답 box를 그대로 예측으로 넣으면 1.0, 클래스를 모두 틀리게 넣으면 0이 나와야 한다.

In [ ]:
def coco_map(gt_objects, pred_objects):
    """mAP@[0.75:0.95]를 계산한다. 정답은 (파일명, K-코드, xywh), 예측은 (파일명, K-코드, xywh, 점수)이다."""
    image_ids = {fn: i + 1 for i, fn in enumerate(sorted({o[0] for o in gt_objects}))}
    categories = sorted({o[1] for o in gt_objects} | {o[1] for o in pred_objects})
    gt = {
        'images': [{'id': i, 'width': IMG_W, 'height': IMG_H} for i in image_ids.values()],
        'annotations': [{'id': k + 1, 'image_id': image_ids[fn], 'category_id': c, 'bbox': list(b),
                         'area': b[2] * b[3], 'iscrowd': 0} for k, (fn, c, b) in enumerate(gt_objects)],
        'categories': [{'id': c} for c in categories],
    }
    if not pred_objects:
        return 0.0
    with contextlib.redirect_stdout(io.StringIO()):  # pycocotools의 진행 메시지 숨김
        coco_gt = COCO()
        coco_gt.dataset = gt
        coco_gt.createIndex()
        coco_dt = coco_gt.loadRes([{'image_id': image_ids[fn], 'category_id': c, 'bbox': list(b), 'score': s}
                                   for fn, c, b, s in pred_objects])
        evaluator = COCOeval(coco_gt, coco_dt, 'bbox')
        evaluator.params.iouThrs = np.linspace(0.75, 0.95, 5)  # 대회 지표의 IoU 기준
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()
    return float(evaluator.stats[0])  # stats[0]: 모든 IoU 기준과 클래스의 AP 평균


eval_objects = clean_objects
wrong_class = {c: train_classes[(i + 1) % len(train_classes)] for i, c in enumerate(train_classes)}
show_table(['예측', 'mAP@[0.75:0.95]'], [
    ['정답 box와 클래스를 그대로 넣음', f'{coco_map(eval_objects, [(*o, 1.0) for o in eval_objects]):.3f}'],
    ['box는 그대로, 클래스를 모두 다른 클래스로 바꿈', f'{coco_map(eval_objects, [(fn, wrong_class[c], b, 1.0) for fn, c, b in eval_objects]):.3f}'],
    ['예측 없음', f'{coco_map(eval_objects, []):.3f}'],
])

**인사이트**
- 정답 box와 클래스를 그대로 넣으면 1.000이다.
- box가 맞아도 클래스가 틀리면 0.000이고, 예측이 없어도 0.000이다.

**결론**
- 이 `coco_map` 함수를 로컬 val 평가에 쓴다.
    - pycocotools 기본 설정(이미지당 최대 검출 100개, 넓이 구간 전체)에서 IoU 기준만 0.75~0.95로 바꿨다.
- 클래스가 틀리면 box가 완벽해도 0점이므로, category_id 체계를 정확히 맞춰야 한다.

### 8-2. box 오차에 대한 민감도
- 정답 box를 오른쪽 아래로 n픽셀씩 옮겨 예측으로 넣고 mAP를 계산한다. (클래스는 그대로)
- 회색 선은 중앙값 크기 box(3-1)를 n픽셀 옮겼을 때의 IoU이다.

In [ ]:
shifts = np.arange(0, 41, 2)
shift_map = [coco_map(eval_objects, [(fn, c, [b[0] + s, b[1] + s, b[2], b[3]], 1.0) for fn, c, b in eval_objects])
             for s in shifts]
median_w, median_h = np.median(clean_boxes[:, 2]), np.median(clean_boxes[:, 3])
shift_iou = [box_iou([0, 0, median_w, median_h], [s, s, median_w, median_h]) for s in shifts]

fig, ax = plt.subplots(figsize=(10, 4.5), layout='constrained')
ax.plot(shifts, shift_map, marker='o', label='mAP@[0.75:0.95]')
ax.plot(shifts, shift_iou, color='gray', linestyle='--', label=f'중앙값 box({median_w:.0f}×{median_h:.0f})의 IoU')
for s, m in zip(shifts, shift_map):
    if s % 4 == 0:
        ax.annotate(f'{m:.2f}', (s, m), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)
for threshold in [0.75, 0.95]:
    ax.axhline(threshold, color='lightgray', linewidth=1)
ax.set_xlabel('정답 box를 x, y 방향으로 각각 옮긴 거리 (픽셀)')
ax.set_ylabel('값 (0~1)')
ax.set_title('box 위치 오차에 따른 mAP@[0.75:0.95] (가로 회색 선 = IoU 기준 0.75, 0.95)')
ax.legend()
save_fig(fig, '8-2_shift_sensitivity')
plt.show()

**인사이트**
- box를 2픽셀 옮겨도 mAP는 거의 1.00이지만, 4픽셀이면 0.87, 8픽셀이면 0.64로 떨어진다.
- 16픽셀이면 0.25, 24픽셀이면 0.07이다.
- 중앙값 크기 box를 8픽셀 옮겼을 때 IoU는 약 0.87인데, mAP는 0.64까지 떨어진다.
    - IoU 0.90, 0.95 기준을 넘지 못하는 box가 생기기 때문이다.

**결론**
- 라벨 box는 원본 기준 2~4픽셀 안으로 정확해야 한다.
    - 2-1에서 그려 넣은 box는 같은 방법으로 라벨 box를 다시 만들었을 때 IoU 중앙값이 0.974였다. 중앙값 크기 box를 x, y로 1~2픽셀씩 옮긴 정도의 차이라 이 기준 안에 든다.
- 모델 평가에서도 IoU 0.75와 0.95의 AP를 따로 기록해, box 위치 정밀도가 점수를 깎는지 확인한다.
- 입력 640과 1280을 비교하는 실험(3-1)과 임의 각도 회전을 쓰지 않는 결정(4-3)도 이 민감도를 근거로 한다.

### 8-3. 제출 category_id 체계
- 제출 CSV의 category_id에 어떤 값을 넣어야 하는지, 대회 문서와 train 라벨을 대조한다.
    - 대회가 준 라벨의 형식을 보는 것이라 원본 763개 annotation으로 본다.

In [ ]:
show_table(['근거', '내용'], [
    ['RULE.md 제출 예시', 'category_id 1, 24, 11, 69 (작은 정수)'],
    ['DATA.md 가공 설명', 'category_id는 각 JSON의 첫 이미지에 있는 dl_idx를 정수로 바꾼 값'],
    ['train 라벨의 category_id', f'{min(train_classes)}~{max(train_classes)} 사이 {len(train_classes)}종'],
    ['category_id = drug_N 숫자', f'{same_code}/{json_count} (1-1)'],
    ['허용 zip 원본의 dl_idx', 'K-코드보다 1 작거나 다른 값 (6-2). 대회 측이 K-코드로 고친 것으로 보임'],
])

**인사이트**
- RULE.md 예시의 category_id는 1, 24, 11, 69처럼 작은 정수이다.
- DATA.md는 category_id를 dl_idx로 만들었다고 하고, train 라벨의 category_id는 1900~41768 사이의 K-코드 숫자이다.
- 채점은 train 라벨과 같은 K-코드로 할 가능성이 크다. RULE.md 예시는 형식을 보여 주는 값으로 보인다.

**결론**
- 제출 category_id에는 K-코드 숫자(예: 16548)를 넣는다.
- YOLO는 클래스 번호가 0부터 연속이어야 하므로, 0~117 인덱스와 K-코드의 매핑 표를 학습 데이터와 함께 저장하고, 제출할 때 K-코드로 되돌린다.
- 첫 기준선 제출 점수가 0에 가까우면 category_id 체계부터 확인한다.

## 9. EDA 결론
- 1~8번의 결과를 항목별 결정 사항으로 정리한다. 괄호 안 번호는 근거가 된 셀의 번호이다.
- Phase A 기준이다. 보충 이미지를 받은 뒤(Phase B) 분할 항목을 확정한다.

| 항목 | 결정 | 근거 |
| --- | --- | --- |
| 라벨 로딩 | `load_annotations`로 알약별 JSON을 이미지 단위로 합쳐 읽음 | 이미지 하나의 라벨이 JSON 2~4개로 흩어져 있음 (1-1) |
| 라벨 정제: 그려 넣기 | 라벨 누락 8장은 빠진 box를 사진에서 그려 넣고, 두 클래스의 box가 똑같은 3장은 틀린 클래스의 box를 box가 없던 알약으로 옮김 | 알약 범위 + 같은 클래스 라벨 여백으로 다시 만든 box가 라벨 box와 IoU 중앙값 0.974 (2-1), 옮길 클래스는 사진으로 확인 (2-2), AI Hub에서 복구할 수 없음 (6-3), 목록 (2-5) |
| 라벨 정제: 수정 | K-018357의 x를 6567에서 567로, K-003351의 x를 185에서 385로, K-027733과 K-035206의 라벨을 서로 바꿈, K-019232의 높이를 241에서 341로, K-003351의 높이를 182에서 192로 고침 (각 1장) | x 후보 중 567만 box가 알약을 감쌈 (2-2), x를 385로 옮기면 box가 옆의 HT 정제를 감쌈 (2-3), 두 클래스 줄에 서로의 알약이 있음 (2-3), 원본 area에 맞춘 높이에서 알약 둘레 여백이 고름 (2-2) |
| area | bbox를 고친 뒤 너비×높이로 다시 계산 | 6개가 원본 area와 다름. 그중 2개는 bbox 높이 오타, 4개는 area 값 오타 (2-2) |
| 입력 크기 | 640으로 시작하고, 1280(원본 해상도)과 같은 조건으로 비교해 확정 (추가 실험 후보). 추론은 학습과 같은 크기 | 입력 640에서 box 짧은 변 최소 62, 중앙값 107픽셀 (3-1), 원본 4픽셀 오차에 mAP 0.87 (8-2), 각인만 다른 알약이 있음 (4-3, 5-3) |
| 데이터 증강: 색, 밝기 | ultralytics 기본값 (`hsv_h=0.015`, `hsv_s=0.7`, `hsv_v=0.4`) | train과 test의 채널 평균 차이 3~7% (1-2) |
| 데이터 증강: 좌우, 상하 반전 | 사용하지 않음 (`fliplr=0`, `flipud=0`) | 실제 사진에 거울상 각인이 없음 (4-3) |
| 데이터 증강: 임의 각도 회전 | 사용하지 않음 (`degrees=0`) | 돌린 box가 헐거워짐, 4픽셀 오차에 mAP 0.87 (8-2) |
| 데이터 증강: 크기 조절, 이동, 모자이크 | ultralytics 기본값 | 작은 객체가 없음 (3-1) |
| 후처리: NMS IoU | 0.7(기본값)로 시작하고 0.5와 val에서 비교 | 실제 box끼리 IoU 최대 0.021, 맞닿은 알약도 box 여백만 겹침 (3-2) |
| 후처리: 최대 검출 수 | 기본값 300 | 낮은 점수 예측도 재현율을 올림 (3-2) |
| 클래스 체계 | train 56개 + 허용 zip 62개 = 118개, K-코드와 0~117 인덱스 매핑 표를 저장 | test에 train에 없는 알약 18종(사진의 39.8%), 그중 17종이 허용 zip의 train에 없는 클래스 (5-3, 6-4, 6-5) |
| 보충 데이터 | 허용 조합 zip 8쌍(TS_1, TS_3~8, VS_1)에서 클래스당 목표 100개, 5,192장(약 8.5GB) | 디스크 23GiB 안에 들어옴 (6-6), 목록 `data/external/aihub_targets.csv` |
| 클래스 불균형 | 기준선은 클래스 가중치와 반복 샘플링 없이 학습. 객체가 적은 K-003483(45개), K-020753(90개)은 val 클래스별 AP로, test 전용 클래스 17개는 그룹 평균 AP로 따로 확인 | 보충 후 최대÷최소 17.0배, 118개 중 94개가 100~199개, 가장 많은 클래스 3.8%, 원본 train·test 전용·허용 zip 추가 클래스의 객체 수 중앙값 152.5·133·143개 (6-7) |
| 보충 데이터 변환 | 키는 그대로 두고 7개 값만 고침: id 3개는 새로 매기고, category_id·categories id·dl_idx = drug_N 숫자, categories name = 제품명. 오류 이미지 163장 제외 | JSON 키 구성이 Kaggle과 같음 45,932/45,933 (6-2), 같은 알약의 JSON에서 값이 다른 키 11개 중 7개가 id와 클래스 표기 (6-2b), 오류 이미지 (6-4) |
| 금지 데이터 | `TL_2_조합.zip`, `TS_2_조합.zip`은 받지 않고, 공용 모듈이 파일 이름으로 막음 | RULE.md, 대회 데이터의 출처 (6-3) |
| 분할 | 조합 단위, val 20%, `make_group_split` (보충 후 Phase B에서 확정) | 같은 조합 = 같은 장면 (4-3), test의 87%가 train에 없는 조합 (5-2), 보충 후 모든 클래스가 15조합 이상 (6-7) |
| 평가 방법 | `coco_map`(pycocotools, IoU 0.75~0.95)으로 val 평가, IoU 0.75와 0.95의 AP도 따로 기록 | 검증 1.000 (8-1), 위치 오차에 민감 (8-2) |
| 제출 category_id | K-코드 숫자 | train 라벨 category_id = K-코드 763/763 (1-1, 8-3) |

**추가 실험 후보**
- 학습 이미지를 180° 돌린 복사본을 넣어, 회전에 강해지는지 비교한다.
    - 같은 조합의 70° 사진은 배치가 180° 돌아가 있다. (4-3)
- 입력 1280(시간이 부족하면 960)을 640 기준선과 같은 분할과 설정으로 학습해 비교한다. (3-1)
    - mAP@[0.75:0.95], IoU 0.95의 AP, 비슷한 알약끼리의 혼동을 함께 보고, 학습 시간이 약 4배 늘어날 가치가 있는지 판단한다.
- 허용 조합 zip에 없는 Hanmi 정제(test 28장)는 단일 데이터를 별도 환경에서 받아 보충한다. (6-11)
- 객체가 적은 K-003483(45개), K-020753(90개)의 val AP가 낮으면, 이 클래스가 든 이미지를 반복해서 넣어 비교한다. (6-7)
- test 전용 클래스 17개의 val 평균 AP가 다른 그룹보다 낮으면, 이 클래스들만 목표 객체 수를 200개로 올려 보충해 비교한다. (6-6, 6-7)
- 학습 뒤 val에서 손실이 큰 이미지를 눈으로 확인해, 갤러리(2-3)에서 놓친 라벨 오류를 찾는다.